# Cách 1: conv ba nhánh, gộp lại khi suy luận (`c1_rep_branches`)

**Câu hỏi.** Lúc train, thay mỗi Conv3x3 trong trunk của e80 bằng ba nhánh conv cộng lại. Khi suy luận gộp chúng về
đúng một Conv3x3. Cách này có tăng PSNR mà không tốn thêm chi phí suy luận không?

**Thiết kế.** Stage, exit (K = 5 / 9 / 15), head, decoder PCA và loss giữ nguyên e80. Chỉ conv trong 7 ResBlock
được thay:

```
lúc train:    x ─┬─ nhánh 1 ─┐
                 ├─ nhánh 2 ─┼─ (+) ─ ReLU ─ ...      (mọi nhánh tuyến tính, không BN)
                 └─ nhánh 3 ─┘
lúc suy luận: x ── Conv3x3 (đã gộp) ─ ReLU ─ ...      = e80, nạp được vào NuSEE thường (strict)
```

Trục K không đổi: model sau khi gộp có đúng tham số, FLOPs và latency của e80 ở mọi K.

| Biến thể | Ba nhánh | Nguồn | Tham số suy luận K = 5 / 9 / 15 | Tham số lúc train |
|---|---|---|---|---|
| `e80` | Conv3x3 (tham chiếu) | | 94,021 / 112,913 / 132,003 | 132,003 |
| `e80rep` | 3x3 + 1x1 + identity | RepVGG (CVPR 2021) | như e80 | 146,787 |
| `e80acb` | 3x3 + 1x3 + 3x1 | ACNet (ICCV 2019) | như e80 | 218,915 |
| `e80ecb` | 3x3 + (1x1 → 3x3, nở 2 lần) + (1x1 → Sobel-x / Sobel-y / Laplacian) | ECBSR (ACM MM 2021) | như e80 | 436,643 |

`e80rep` dùng `RepConv` có sẵn của `nuse_e`. Bản RepConv trước đây (`rep7`) được thử trên exit 2/5/9/15 của v2,
chưa thử trên exit của e80, nên ở đây train lại để so cùng điều kiện.

**Đã biết trước.** rep7 +0.03 dB, rep8 −0.01 dB so với bản không rep. Hai lần train cùng seed lệch nhau tới 0.01 dB.
Kỳ vọng thấp: re-param chỉ đổi đường đi của gradient lúc train, không tăng sức biểu diễn của model đã gộp.
ECB là ứng viên đáng thử nhất, vì nhánh Sobel / Laplacian khớp với dạng của vài vector PCA đầu
(mặt dốc theo hai hướng, dạng yên ngựa trong ô 4×4).

**Coi là có ích khi** PSNR ở K = 15 cao hơn e80 quá 0.01 dB, các K nhỏ không tụt, và cột ràng buộc trong
`comparison/constraints.csv` ghi đạt hết. Train ECB chậm hơn e80 vì nhánh 1x1 → 3x3 nở gấp đôi kênh.

**Hai loại LR.** Train trên cả LR average pool và LR bicubic (`METHODS` trong `experiment.py`): pha A train mọi biến thể trên
LR average pool, pha B train mọi biến thể trên LR bicubic (`PHASE_B_TOP = len(VARIANTS)`, không chỉ 2 bản tốt nhất).
So mỗi biến thể với e80 cùng loại LR. Output v3 có sẵn run tham chiếu bicubic (v2, e80, d8).

## File

| File | Nội dung |
|---|---|
| `experiment.py` | Biến thể và thiết lập riêng; pipeline `nuse_e` đọc file này qua `--experiment` |
| `rep_branches/branches.py` | `ACBConv`, `ECBConv` và phép gộp (`fuse()`) |
| `rep_branches/model.py` | `RepBranchNuSEE`: NuSEE với block ba nhánh |
| `test_rep_branches.py` | Gộp đúng cả ở viền ảnh; model đã gộp = e80; A(SR) = y; exit nhỏ không đọc phần sau |
| `kaggle_c1_rep_branches.ipynb` | Notebook Kaggle tự chứa, sinh bởi `experiments/build_notebooks.py` |

## Chạy trên máy (từ gốc repo)

```bash
python -m unittest experiments/c1_rep_branches/test_rep_branches.py
python -m nuse_e.pipeline setup --out outputs/c1 --experiment experiments/c1_rep_branches --train-hr <DIV2K_train_HR> --val-hr <DIV2K_valid_HR> --resume-from <output v3>
python -m nuse_e.pipeline stats test screen train curves select evaluate cost report figures --out outputs/c1
```


## Chạy trên Kaggle

Notebook này tự chứa toàn bộ code: chỉ cần upload đúng file `.ipynb` này (File → Import Notebook) rồi thêm dữ liệu.

1. **Add Input**
   - DIV2K (`DIV2K_train_HR`, `DIV2K_valid_HR`). Bắt buộc.
   - Output của notebook v3 (`vidubatman/nuse-e-v3`). Nên có: run tham chiếu (v2, e80, d8) có `DONE` sẵn nên không
     train lại, `stats.pt` dùng lại. Không có thì notebook train cả các run tham chiếu (thêm vài giờ).
   - `nuse_old_runs_kaggle.zip` (checkpoint baseline cũ). Nên có: trần latency đo lại trên cùng GPU, có hàng baseline trong bảng.
   - Set5 / Set14 / B100 / Urban100 / Manga109. Tuỳ chọn.
2. **Settings**: GPU T4 ×2, Internet bật (để `pip install lpips torchmetrics torch-fidelity`; không có thì bỏ LPIPS / FID).
3. **Save Version → Save & Run All (Commit)**, không chạy trong phiên tương tác. Bản commit chạy trên máy của Kaggle,
   đóng tab hay tắt máy không ảnh hưởng; xong thì output nằm ở tab Output của version.
   Muốn kiểm notebook trước: đặt `SMOKE = True` ở ô dưới (vài bước train, vài phút).

**Nếu train chưa xong trong một lần** (Kaggle giới hạn 12 giờ mỗi lần chạy): gần hết `TRAIN_HOURS`, run đang train lưu
`last.pt` rồi dừng, notebook bỏ qua bước đánh giá và kết thúc bình thường để Kaggle lưu output. Khi đó: Add Input →
Your Work → output version vừa chạy của chính notebook này, rồi Save & Run All lần nữa. Notebook tìm mọi output cũ trong
Input, lấy bản tiến xa nhất của từng run và train tiếp từ epoch đã dừng (dữ liệu sinh theo seed của epoch nên kết quả
như chạy một mạch). Version bị lỗi hay bị Stop giữa chừng cũng làm như vậy, nếu tab Output của nó có `out/runs/`.

Output nằm trong `/kaggle/working/out`: `comparison/report.md`, `comparison/variants.csv`, `comparison/constraints.csv`,
`comparison/figures/`, `runs/<average_pool | bicubic>/<model>/`.


In [ ]:
import os
import time
from pathlib import Path

PROJECT = 'c1_rep_branches'
SMOKE = os.environ.get("NUSE_SMOKE") == "1"   # True: chạy thử cực ngắn (2 bước train, ít ảnh) để kiểm notebook
GPUS = None                                   # ví dụ "0,1"; None: mọi GPU
INPUT = Path(os.environ.get("NUSE_INPUT", "/kaggle/input"))            # nơi tìm DIV2K, output v3, checkpoint cũ
WORK = Path(os.environ.get("NUSE_WORK", "/kaggle/working"))
CODE, OUT = WORK / "code", WORK / "out"
CACHE = os.environ.get("NUSE_CACHE", "/tmp/nuse_cache")                # cache ảnh: không lưu vào output

# Kaggle dừng mỗi lần chạy sau 12 giờ. Train dừng sau TRAIN_HOURS tính từ ô này, chừa thời gian đánh giá và lưu output.
TRAIN_HOURS = float(os.environ.get("NUSE_TRAIN_HOURS", 10.5))
START = time.time()


In [ ]:
# Code nhúng: sinh bởi experiments/build_notebooks.py từ code local. Đừng sửa tay ô này.
FILES = {
    'nuse_e/__init__.py': '"""NuSE-E: siêu phân giải ×4 bằng PCA trong null space, một model nhiều exit theo số hệ số K.\n\nCác package con, xếp từ tầng thấp lên tầng cao (tầng sau chỉ import tầng trước):\n\n    core/         hằng số, seed, ghi JSON / CSV\n    operators.py  toán tử đo A (average pool, bicubic) và giả nghịch đảo A⁺\n    data/         đọc ảnh, cache ảnh train, bộ lấy crop\n    nullspace/    cơ sở Helmert, thống kê PCA và Wiener, quy tắc suy kiến trúc từ phổ PCA\n    models/       NuSE-E, các khối của trunk, baseline, tạo và nạp model\n    evaluation/   PSNR, SSIM, LPIPS, FID, đánh giá trên tập ảnh, tham số, GFLOPs, latency\n    training/     vòng train một run (python -m nuse_e.training.run <config.json>)\n    pipeline/     các bước của thí nghiệm (python -m nuse_e.pipeline <bước>)\n    config.py     thiết lập thí nghiệm\n\nXem README.md.\n"""\n',
    'nuse_e/config.py': '"""Thiết lập thí nghiệm NuSE-E (trước đây là ô cấu hình của notebook).\n\nSửa giá trị ở đây rồi chạy lại bước cần thiết. Đường dẫn dữ liệu không nằm ở đây: truyền qua\n`python -m nuse_e.pipeline setup ...` (xem README.md).\n"""\nfrom types import SimpleNamespace\n\n# ---- Thí nghiệm ----\nMETHODS = ["average_pool", "bicubic"]               # cách tạo LR\nBASELINES = ["SPANV2", "DISP", "PLKSR-Tiny"]        # chỉ dùng checkpoint cũ / số đã báo cáo (LR average pool)\nOLD_RUNS = "reuse"   # "reuse": baseline LR average_pool lấy từ checkpoint cũ nếu có; "retrain": train lại tất cả\nSHOW_OLD_PLUGIN = True                              # thêm dòng "backbone + NuSE (k=8)" của thí nghiệm cũ\n\n# ---- Biến thể NuSE-E. Cùng quy tắc suy kiến trúc từ phổ PCA; chỉ khác các siêu tham số mà quy tắc không quyết\n# định: tổng số block, độ rộng, kiểu block, mục tiêu năng lượng, trọng số loss alpha.\n# Thứ tự trong dict = thứ tự ưu tiên train. "NuSE-E" (không có hậu tố) là bản v2, luôn có mặt để đối chiếu.\nVARIANTS = {\n    "d8":     dict(total_blocks=8, width=32, block="plain"),                                   # sâu thêm 1 block\n    "rep7":   dict(total_blocks=7, width=32, block="rep"),                                     # RepConv, suy luận = v2\n    "w40":    dict(total_blocks=5, width=40, block="plain"),                                   # rộng hơn, nông hơn\n    "rep8":   dict(total_blocks=8, width=32, block="rep"),\n    "lk7":    dict(total_blocks=7, width=32, block="lk7"),                                     # depthwise 7x7 trong block\n    "e80":    dict(total_blocks=7, width=32, block="plain", energy_targets=(0.8, 0.9, 1.0)),   # bỏ exit K=2 (bản chính)\n    "top":    dict(total_blocks=7, width=32, block="plain", alpha="top"),                      # loss chỉ ở K lớn nhất\n    "w36":    dict(total_blocks=6, width=36, block="plain"),\n    "w40rep": dict(total_blocks=5, width=40, block="rep"),\n}\nPHASE_B_TOP = 2            # xong LR average pool thì train thêm trên LR bicubic cho N biến thể tốt nhất\nSELECT_TOL_DB = 0.01       # chọn biến thể: chênh PSNR K lớn nhất dưới ngưỡng này coi là hoà, lấy bản ít tham số hơn\n                           # (hai lần train độc lập cùng seed lệch nhau tới 0.01 dB)\n\n# ---- Ràng buộc cứng: kiểm trước khi train (tham số, latency) và sau khi train (đơn điệu theo K, khớp LR)\nPARAM_CAP = 156856         # tham số ở K lớn nhất <= DISP + NuSE k=8\nLATENCY_REF = "DISP + NuSE k=8 (cũ)"   # trần latency @LR 256² fp32: đo lại từ checkpoint cũ nếu có\nLATENCY_CAP_MS = 8.44      # dùng khi không có checkpoint cũ (DISP + NuSE k=8, T4)\nCONSISTENCY_TOL = 1e-5     # max|A(SR) - y| cho phép\n\n# ---- Giao thức train (giống các run baseline cũ) ----\nEPOCHS, BATCH, REPEAT = 100, 128, 20                # 800 ảnh × 20 / 128 = 125 iteration mỗi epoch\nLR, MIN_LR, GRAD_CLIP, SEED = 1e-3, 1e-6, 1.0, 2026\nVAL_EVERY, CHECKPOINT_EVERY = 5, 10                 # đánh giá đủ 100 ảnh (chọn best.pt) mỗi 5 epoch\nQUICK_VAL_IMAGES, QUICK_VAL_CROP = 10, 480          # mỗi epoch: 10 crop 480×480 để theo dõi tiến độ\nAMP = False                                         # phải giống nhau cho mọi model\nWORKERS = 4\nMAX_HOURS = None                                    # ví dụ 11.5: dừng nhận run mới sau chừng ấy giờ\n\n# ---- Quy tắc kiến trúc NuSE-E (thiết kế v2, mục 4) ----\nDESIGN = dict(energy_targets=(0.5, 0.8, 0.9, 1.0), total_blocks=7, r2_min=0.05, rel_tol=1e-6, min_share=0.5,\n              width=32, block="plain", alpha="uniform")   # = bản v2 (tham chiếu)\nSTATS_REPEAT, STATS_FOLDS = 4, 5\n\n# ---- Đánh giá ----\nUSE_LPIPS, USE_FID = True, True                     # cần gói lpips, torchmetrics, torch-fidelity\nBENCHMARK_NAMES = ["Set5", "Set14", "B100", "BSD100", "Urban100", "Manga109"]\nSAVE_SR_IMAGES = False\n\n# ---- Latency ----\nLATENCY_SIZES = [(128, 128), (256, 256), (339, 510)]\nLATENCY_PRECISIONS = ["fp32", "fp16"]\nWARMUP, ITERS, ROUNDS = 30, 300, 10\nCPU_LATENCY_SIZE, CPU_ITERS = (128, 128), 20\nFLOPS_SIZE = (256, 256)\n\n# ---- Checkpoint của thí nghiệm cũ (tìm trong --old-runs) ----\nOLD_NAMES = {"SPANV2": "spanv2_x4", "DISP": "disp_x4", "PLKSR-Tiny": "plksr_tiny_x4",\n             "SPANV2 + NuSE k=8 (cũ)": "spanv2_nuse_x4_k8", "DISP + NuSE k=8 (cũ)": "disp_nuse_x4_k8",\n             "PLKSR-Tiny + NuSE k=8 (cũ)": "plksr_tiny_nuse_x4_k8"}\n\n# Số đã báo cáo của các run cũ (LR average_pool, DIV2K valid 100 ảnh). Dùng khi không có checkpoint cũ.\nOLD_REPORTED = {\n    \'DISP\': {\'rgb_psnr\': 27.520466, \'rgb_ssim\': 0.782759, \'lpips\': 0.344047, \'lr_max_abs\': 0.10915189981460571, \'params\': 163792, \'gflops\': 21.3868, \'gpu\': \'Tesla T4\', \'ms_fp32_339x510\': 24.042, \'ms_fp16_339x510\': 14.673},\n    \'DISP + NuSE\': {\'rgb_psnr\': 28.111805, \'rgb_ssim\': 0.799086, \'lpips\': 0.284612, \'lr_max_abs\': 2.384185791015625e-07, \'params\': 156856, \'gflops\': 20.5311, \'gpu\': \'Tesla T4\', \'ms_fp32_339x510\': 24.651, \'ms_fp16_339x510\': 15.698},\n    \'PLKSR-Tiny\': {\'rgb_psnr\': 28.486712, \'rgb_ssim\': 0.809117, \'lpips\': 0.272896, \'lr_max_abs\': 0.0018892884254455566, \'params\': 2370544, \'gflops\': 310.2694, \'gpu\': \'NVIDIA GeForce RTX 3090\', \'ms_fp32_339x510\': 76.988, \'ms_fp16_339x510\': 51.477},\n    \'PLKSR-Tiny + NuSE\': {\'rgb_psnr\': 28.548999, \'rgb_ssim\': 0.811089, \'lpips\': 0.267085, \'lr_max_abs\': 2.384185791015625e-07, \'params\': 2356696, \'gflops\': 308.5078, \'gpu\': \'NVIDIA GeForce RTX 3090\', \'ms_fp32_339x510\': 78.418, \'ms_fp16_339x510\': 50.007},\n    \'SPANV2\': {\'rgb_psnr\': 27.989892, \'rgb_ssim\': 0.796165, \'lpips\': 0.29634, \'lr_max_abs\': 0.050102829933166504, \'params\': 140944, \'gflops\': 18.3627, \'gpu\': \'NVIDIA GeForce RTX 3090\', \'ms_fp32_339x510\': 9.12, \'ms_fp16_339x510\': 6.987},\n    \'SPANV2 + NuSE\': {\'rgb_psnr\': 28.044281, \'rgb_ssim\': 0.797652, \'lpips\': 0.288856, \'lr_max_abs\': 2.384185791015625e-07, \'params\': 139000, \'gflops\': 18.1613, \'gpu\': \'NVIDIA GeForce RTX 3090\', \'ms_fp32_339x510\': 9.659, \'ms_fp16_339x510\': 7.892},\n}\n\n\ndef load(smoke=False, only=None, overrides=None):\n    """Bản sao các thiết lập trên. `smoke`: chạy thử cực ngắn để kiểm code. `only`: chỉ giữ các biến thể này.\n    `overrides`: thiết lập viết hoa của một thí nghiệm (file experiment.py, xem experiments/README.md) thay giá trị ở đây."""\n    c = SimpleNamespace(**{k: v for k, v in globals().items() if k.isupper()})\n    c.__dict__.update({k: v for k, v in (overrides or {}).items() if k.isupper()})\n    variants = c.VARIANTS = dict(c.VARIANTS)\n    if only is not None:\n        c.VARIANTS = {k: v for k, v in variants.items() if k in only}\n    if smoke:\n        c.EPOCHS, c.BATCH, c.REPEAT, c.VAL_EVERY, c.WORKERS, c.STATS_REPEAT = 2, 2, 1, 1, 0, 1\n        c.QUICK_VAL_IMAGES, c.QUICK_VAL_CROP = 2, 64\n        c.LATENCY_SIZES, c.WARMUP, c.ITERS, c.ROUNDS, c.CPU_ITERS = [(32, 32)], 1, 2, 1, 2\n        c.USE_LPIPS, c.USE_FID = False, False\n        if only is None and not overrides:\n            c.VARIANTS = {k: variants[k] for k in ("d8", "rep7", "e80")}\n        if "PHASE_B_TOP" not in (overrides or {}):          # thí nghiệm: chạy thử đúng các run pha B sẽ train\n            c.PHASE_B_TOP = 1\n    c.MAX_ITERS = 2 if smoke else None\n    c.SMOKE = smoke\n    c.NUSE = ["NuSE-E", *[f"NuSE-E/{v}" for v in c.VARIANTS]]   # "NuSE-E" = bản v2; "NuSE-E/<tên>" = biến thể\n    c.MODELS = c.NUSE + c.BASELINES\n    return c\n',
    'nuse_e/core/__init__.py': '"""Hằng số và tiện ích dùng chung, không phụ thuộc phần nào khác của package."""\n',
    'nuse_e/core/constants.py': '"""Hằng số dùng chung: hệ số phóng, đuôi file ảnh, hai cách tạo LR."""\nS = 4                                   # scale\n\n\nIMAGE_EXT = {".png", ".jpg", ".jpeg", ".bmp"}\n\n\nMETHODS = ("average_pool", "bicubic")\n',
    'nuse_e/core/io.py': '"""Seed, ghi JSON / CSV."""\nimport csv\nimport json\nimport math\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed % 2**32)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n\n    def safe(v):\n        if isinstance(v, float) and not math.isfinite(v):\n            return str(v)\n        if isinstance(v, dict):\n            return {str(k): safe(x) for k, x in v.items()}\n        if isinstance(v, (list, tuple)):\n            return [safe(x) for x in v]\n        if isinstance(v, Path):\n            return str(v)\n        if isinstance(v, torch.Tensor):\n            return safe(v.tolist())\n        return v\n    path.write_text(json.dumps(safe(value), indent=2, ensure_ascii=False) + "\\n")\n\n\ndef write_csv(path, rows):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    keys = list(rows[0])\n    for row in rows[1:]:\n        keys += [k for k in row if k not in keys]\n    with path.open("w", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=keys)\n        writer.writeheader()\n        writer.writerows(rows)\n',
    'nuse_e/data/__init__.py': '"""Dữ liệu: đọc ảnh (images), cache ảnh train dạng uint8 phẳng (cache), bộ lấy crop để train (loaders)."""\n',
    'nuse_e/data/cache.py': '"""Cache ảnh train: giải mã một lần vào một file uint8 phẳng, crop đọc thẳng từ memmap."""\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image\n\ndef _decode_flat(path):\n    with Image.open(path) as image:\n        return np.asarray(image.convert("RGB"), dtype=np.uint8).tobytes(), image.size[1], image.size[0]\n\n\ndef build_flat_cache(files, path, workers=4):\n    """Decode every training image once into <path> (raw HWC uint8, back to back) + <path>.json index."""\n    from concurrent.futures import ProcessPoolExecutor\n    path, index_path = Path(path), Path(str(path) + ".json")\n    files = [Path(f) for f in files]\n    if index_path.exists() and path.exists():\n        index = json.loads(index_path.read_text())\n        if [e["name"] for e in index] == [f.stem for f in files]:\n            return path\n    index, offset = [], 0\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("wb") as out, ProcessPoolExecutor(max_workers=max(1, workers)) as pool:\n        for f, (raw, h, w) in zip(files, pool.map(_decode_flat, [str(f) for f in files], chunksize=4)):\n            out.write(raw)\n            index.append({"name": f.stem, "offset": offset, "h": h, "w": w})\n            offset += len(raw)\n    index_path.write_text(json.dumps(index))\n    return path\n\n\nclass FlatImages:\n    def __init__(self, path):\n        self.index = json.loads(Path(str(path) + ".json").read_text())\n        self.data = np.memmap(str(path), dtype=np.uint8, mode="r")\n\n    def __len__(self):\n        return len(self.index)\n\n    def size(self, i):\n        e = self.index[i]\n        return e["h"], e["w"]\n\n    def crop(self, i, top, left, c):\n        e = self.index[i]\n        view = self.data[e["offset"]:e["offset"] + e["h"] * e["w"] * 3].reshape(e["h"], e["w"], 3)\n        return torch.from_numpy(np.ascontiguousarray(view[top:top + c, left:left + c])).permute(2, 0, 1)\n',
    'nuse_e/data/images.py': '"""Đọc, ghi ảnh và tập ảnh valid."""\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\n\nfrom ..core.constants import IMAGE_EXT, S\n\ndef image_files(folder):\n    return sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in IMAGE_EXT)\n\n\ndef load_rgb(path):\n    with Image.open(path) as image:\n        return torch.from_numpy(np.array(image.convert("RGB"), dtype=np.uint8, copy=True)).permute(2, 0, 1)\n\n\ndef to_bmp(args):\n    """(source, folder) -> cached BMP path; PNG 2K decoding is slow, BMP reads almost instantly."""\n    source, folder = args\n    target = Path(folder) / (Path(source).stem + ".bmp")\n    if not target.exists():\n        with Image.open(source) as image:\n            image.convert("RGB").save(target.with_suffix(".partial"), format="BMP")\n        target.with_suffix(".partial").rename(target)\n    return str(target)\n\n\ndef save_png(path, image):\n    """Clamp/round for viewing only; metrics never use this."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    pixels = image.detach().cpu().clamp(0, 1).mul(255).round().byte().permute(1, 2, 0).numpy()\n    Image.fromarray(pixels).save(path)\n\n\nclass ValImages(Dataset):\n    """Full images as uint8 (to keep RAM low), cropped bottom/right to a multiple of 4."""\n\n    def __init__(self, files):\n        self.files = list(files)\n\n    def __len__(self):\n        return len(self.files)\n\n    def __getitem__(self, index):\n        hr = load_rgb(self.files[index])\n        h, w = hr.shape[-2] // S * S, hr.shape[-1] // S * S\n        return hr[:, :h, :w].contiguous(), self.files[index].stem\n\n\ndef as_float(hr_uint8, device):\n    return hr_uint8[None].to(device).float().div_(255)\n',
    'nuse_e/data/loaders.py': '"""Bộ lấy crop HR ngẫu nhiên để train (crop, lật, xoay, lặp)."""\nimport math\nimport random\n\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import DataLoader\nfrom torch.utils.data import Dataset\n\nclass TrainCrops(Dataset):\n    """HR crops (default 256 = LR 64 x 4); LR is generated on the GPU by `degrade`.\n\n    Same sampling as the repo DIV2KDataset: per-item RNG seeded by (seed, epoch, index),\n    flips + 90-degree rotations, `repeat` crops per image per epoch.\n    """\n\n    def __init__(self, files, crop=256, repeat=20, augment=True, seed=2026):\n        self.files, self.crop, self.repeat, self.augment, self.seed = list(files), crop, repeat, augment, seed\n        self.epoch = 0\n\n    def __len__(self):\n        return len(self.files) * self.repeat\n\n    def set_epoch(self, epoch):\n        self.epoch = epoch\n\n    def __getitem__(self, index):\n        path = self.files[index % len(self.files)]\n        rng = random.Random(self.seed + self.epoch * len(self) + index)\n        with Image.open(path) as image:\n            image = image.convert("RGB")\n            w, h = image.size\n            left, top = rng.randrange(w - self.crop + 1), rng.randrange(h - self.crop + 1)\n            image = image.crop((left, top, left + self.crop, top + self.crop))\n            if self.augment:\n                if rng.random() < 0.5:\n                    image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)\n                if rng.random() < 0.5:\n                    image = image.transpose(Image.Transpose.FLIP_TOP_BOTTOM)\n                turns = rng.randrange(4)\n                if turns:\n                    image = image.transpose({1: Image.Transpose.ROTATE_90, 2: Image.Transpose.ROTATE_180,\n                                             3: Image.Transpose.ROTATE_270}[turns])\n            hr = torch.from_numpy(np.array(image, dtype=np.uint8, copy=True)).permute(2, 0, 1)\n        return hr.float().div_(255), index % len(self.files)\n\n\nclass RamCropLoader:\n    """Same sampling as TrainCrops (per-item RNG seeded by (seed, epoch, index), flips + rotations,\n    `repeat` crops per image per epoch); yields uint8 batches (B,3,crop,crop) straight from RAM."""\n\n    def __init__(self, images, batch, seed, crop=256, repeat=20, augment=True):\n        self.images, self.batch, self.seed, self.crop, self.repeat, self.augment = images, batch, seed, crop, repeat, augment\n        self.epoch = 0\n\n    def set_epoch(self, epoch):\n        self.epoch = epoch\n\n    def __len__(self):\n        return math.ceil(len(self.images) * self.repeat / self.batch)\n\n    def item(self, index):\n        i = index % len(self.images)\n        rng = random.Random(self.seed + self.epoch * len(self.images) * self.repeat + index)\n        h, w = self.images.size(i)\n        left, top = rng.randrange(w - self.crop + 1), rng.randrange(h - self.crop + 1)\n        hr = self.images.crop(i, top, left, self.crop)\n        if self.augment:\n            if rng.random() < 0.5:\n                hr = hr.flip(-1)\n            if rng.random() < 0.5:\n                hr = hr.flip(-2)\n            turns = rng.randrange(4)\n            if turns:\n                hr = torch.rot90(hr, turns, (-2, -1))\n        return hr.contiguous(), i\n\n    def __iter__(self):\n        n = len(self.images) * self.repeat\n        order = torch.randperm(n, generator=torch.Generator().manual_seed(self.seed + self.epoch)).tolist()\n        for start in range(0, n, self.batch):\n            items = [self.item(j) for j in order[start:start + self.batch]]\n            yield torch.stack([hr for hr, _ in items]), torch.tensor([i for _, i in items])\n\n\ndef train_loader(files, *, batch, workers, seed, crop=256, repeat=20, augment=True):\n    data = TrainCrops(files, crop, repeat, augment, seed)\n    loader = DataLoader(data, batch_size=batch, shuffle=True, num_workers=workers, pin_memory=True,\n                        drop_last=False, persistent_workers=workers > 0,\n                        generator=torch.Generator().manual_seed(seed))\n    return data, loader\n',
    'nuse_e/evaluation/__init__.py': '"""Đánh giá: metric ảnh (metrics), LPIPS và FID (perceptual), đánh giá trên tập ảnh (evaluate), chi phí (cost)."""\n',
    'nuse_e/evaluation/cost.py': '"""Chi phí: tham số, GFLOPs, latency GPU / CPU."""\nimport time\n\nimport torch\n\nfrom ..models.nuse_e import NuSEE\nfrom ..models.registry import budgets_of\n\ndef count_params(model, k=None):\n    if isinstance(model, NuSEE):\n        return model.params_for(k)\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\n\ndef gflops(model, size, device, k=None):\n    from torch.utils.flop_counter import FlopCounterMode\n    x = torch.rand(1, 3, *size, device=device)\n    with FlopCounterMode(display=False) as counter, torch.no_grad():\n        model(x, k) if isinstance(model, NuSEE) else model(x)\n    return counter.get_total_flops() / 1e9\n\n\n@torch.no_grad()\ndef time_cuda(fns, precision, warmup=30, iters=300, rounds=10):\n    """CUDA events, batch 1, interleaved over `rounds` so clock drift affects every entry alike."""\n    times = {key: [] for key in fns}\n    memory = {}\n    with torch.autocast("cuda", dtype=torch.float16, enabled=precision == "fp16"):\n        for key, fn in fns.items():\n            for _ in range(warmup):\n                fn()\n            torch.cuda.synchronize()\n            torch.cuda.reset_peak_memory_stats()\n            base = torch.cuda.memory_allocated()\n            fn()\n            torch.cuda.synchronize()\n            memory[key] = (torch.cuda.max_memory_allocated() - base) / 2 ** 20\n        for _ in range(rounds):\n            for key, fn in fns.items():\n                events = [(torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True))\n                          for _ in range(max(1, iters // rounds))]\n                for start, end in events:\n                    start.record()\n                    fn()\n                    end.record()\n                torch.cuda.synchronize()\n                times[key] += [s.elapsed_time(e) for s, e in events]\n    return times, memory\n\n\n@torch.no_grad()\ndef time_cpu(fn, iters=20):\n    fn()\n    out = []\n    for _ in range(iters):\n        t = time.perf_counter()\n        fn()\n        out.append((time.perf_counter() - t) * 1e3)\n    return out\n\n\ndef stage_functions(label, model, x):\n    """Entries to time: total for every model; NuSE-E also split into predictor and decoder + consistency."""\n    fns = {}\n    for k in budgets_of(model):\n        name = label if k is None else f"{label} K={k}"\n        if k is None:\n            fns[(name, "total")] = (lambda m=model: m(x))\n        else:\n            coeffs = model.coefficients(x, k)\n            fns[(name, "predictor")] = (lambda m=model, k=k: m.coefficients(x, k))\n            fns[(name, "decoder + consistency")] = (lambda m=model, k=k, c=coeffs: m.consistency(m.detail(c, k), x))\n            fns[(name, "total")] = (lambda m=model, k=k: m(x, k))\n    return fns\n',
    'nuse_e/evaluation/evaluate.py': '"""Đánh giá một model trên một tập ảnh: metric từng ảnh và trung bình."""\nimport statistics\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom ..core.constants import S\nfrom ..data.images import as_float, save_png\nfrom .metrics import psnr, rgb_mse, ssim, to_y\nfrom ..models.nuse_e import NuSEE\nfrom ..operators import degrade, make_operator\n\n@torch.no_grad()\ndef evaluate(model, images, method, device, *, k=None, lpips=None, fid=None, fid_real=False,\n             save_dir=None, amp=False):\n    """Per-image metrics on full images. SR is never clamped for PSNR/SSIM (raw float RGB)."""\n    op = make_operator(method)\n    rows = []\n    model.eval()\n    for hr_cpu, name in images:\n        hr = as_float(hr_cpu, device)\n        lr = degrade(hr, method)\n        with torch.autocast(device.type, dtype=torch.float16, enabled=amp and device.type == "cuda"):\n            sr = model(lr, k) if isinstance(model, NuSEE) else model(lr)\n        sr = sr.float()\n        back = op.forward(sr)\n        row = {"name": name, "rgb_psnr": psnr(sr, hr).item(), "rgb_ssim": ssim(sr, hr).item(),\n               "y_psnr_shave4": psnr(to_y(sr), to_y(hr), S).item(), "y_ssim_shave4": ssim(to_y(sr), to_y(hr), S).item(),\n               "rgb_psnr_uint8": psnr(sr.clamp(0, 1).mul(255).round().div(255), hr).item(),\n               "lr_psnr": psnr(back, lr).item(), "lr_max_abs": (back - lr).abs().max().item(),\n               "sr_below0": (sr < 0).float().mean().item(), "sr_above1": (sr > 1).float().mean().item()}\n        if lpips is not None:\n            row["lpips"] = lpips(sr, hr).item()\n        if fid is not None:\n            fid.update(sr, real=False)\n            if fid_real:\n                fid.update(hr, real=True)\n        if isinstance(model, NuSEE):\n            coeffs = model.coefficients(lr, k)\n            cstar = model.decoder.targets(model.residual_blocks(hr))[:, :, :k]\n            oracle = model.consistency(model.detail(cstar.flatten(1, 2), k), lr)\n            row["oracle_psnr"] = psnr(oracle, hr).item()\n            row["truncation_mse"] = rgb_mse(oracle, hr).item()\n            row["coefficient_image_mse"] = ((coeffs.reshape_as(cstar) - cstar).double().square().sum() / hr.numel()).item()\n        if save_dir is not None:\n            save_png(Path(save_dir) / f"{name}.png", sr[0])\n        rows.append(row)\n    summary = {key: statistics.fmean(r[key] for r in rows) for key in rows[0] if key != "name"}\n    summary["lr_max_abs"] = max(r["lr_max_abs"] for r in rows)\n    if fid is not None:\n        summary["fid_patch256"] = fid.compute_and_reset()\n    summary["images"] = len(rows)\n    return summary, rows\n\n\n@torch.no_grad()\ndef evaluate_bicubic_upsampling(images, method, device, lpips=None, fid=None, fid_real=False):\n    class Up(nn.Module):\n        def forward(self, lr):\n            return F.interpolate(lr, scale_factor=S, mode="bicubic", align_corners=False).clamp(0, 1)\n    return evaluate(Up().to(device), images, method, device, lpips=lpips, fid=fid, fid_real=fid_real)\n',
    'nuse_e/evaluation/metrics.py': '"""PSNR, SSIM (RGB hoặc kênh Y, có shave)."""\nimport torch\nfrom torch.nn import functional as F\n\ndef _crop(a, b, shave):\n    if shave:\n        a, b = a[..., shave:-shave, shave:-shave], b[..., shave:-shave, shave:-shave]\n    return a, b\n\n\ndef rgb_mse(sr, hr, shave=0):\n    sr, hr = _crop(sr, hr, shave)\n    return (sr.double() - hr.double()).square().flatten(1).mean(1)\n\n\ndef psnr(sr, hr, shave=0):\n    return 10 * torch.log10(1.0 / rgb_mse(sr, hr, shave))\n\n\ndef ssim(sr, hr, shave=0):\n    """Gaussian 11x11, sigma 1.5, valid window, population covariance (same as the repo)."""\n    sr, hr = _crop(sr, hr, shave)\n    sr, hr = sr.float(), hr.float()\n    size = min(11, *sr.shape[-2:])\n    size = size if size % 2 else size - 1\n    axis = torch.arange(size, device=sr.device, dtype=sr.dtype) - size // 2\n    g = torch.exp(-axis.square() / (2 * 1.5 ** 2))\n    g = g / g.sum()\n    window = (g[:, None] * g[None, :]).expand(sr.shape[1], 1, size, size)\n\n    def smooth(x):\n        return F.conv2d(x, window, groups=sr.shape[1])\n    mx, my = smooth(sr), smooth(hr)\n    vx = (smooth(sr.square()) - mx.square()).clamp_min(0)\n    vy = (smooth(hr.square()) - my.square()).clamp_min(0)\n    cov = smooth(sr * hr) - mx * my\n    c1, c2 = 0.01 ** 2, 0.03 ** 2\n    return (((2 * mx * my + c1) * (2 * cov + c2)) / ((mx.square() + my.square() + c1) * (vx + vy + c2))).flatten(1).mean(1)\n\n\ndef to_y(x):\n    """uint8-rounded RGB -> Y (BT.601), range [0,1]."""\n    x = x.clamp(0, 1).mul(255).round().div(255)\n    return ((65.481 * x[:, 0] + 128.553 * x[:, 1] + 24.966 * x[:, 2] + 16) / 255)[:, None]\n',
    'nuse_e/evaluation/perceptual.py': '"""LPIPS (AlexNet) và FID trên patch 256×256."""\nimport torch\nfrom torch import nn\n\nclass LPIPSMetric(nn.Module):\n    """LPIPS v0.1 AlexNet, clamp to [0,1] then [-1,1]."""\n\n    def __init__(self):\n        super().__init__()\n        import lpips\n        with torch.random.fork_rng(devices=[]):\n            self.net = lpips.LPIPS(net="alex", version="0.1", verbose=False)\n        self.net.requires_grad_(False)\n        self.eval()\n\n    @torch.no_grad()\n    def forward(self, sr, hr):\n        return self.net(sr.float().clamp(0, 1) * 2 - 1, hr.float().clamp(0, 1) * 2 - 1).flatten()\n\n\nclass PatchFID:\n    """FID on non-overlapping 256x256 patches (Inception-v3 pool 2048), real features kept across models."""\n\n    def __init__(self, device, patch=256):\n        from torchmetrics.image.fid import FrechetInceptionDistance\n        self.metric = FrechetInceptionDistance(feature=2048, normalize=True, reset_real_features=False).to(device)\n        self.patch, self.has_real = patch, False\n\n    def patches(self, x):\n        p = self.patch\n        h, w = x.shape[-2] // p * p, x.shape[-1] // p * p\n        if h == 0 or w == 0:\n            return x.clamp(0, 1)\n        x = x[..., :h, :w].clamp(0, 1)\n        return x.unfold(2, p, p).unfold(3, p, p).permute(0, 2, 3, 1, 4, 5).reshape(-1, 3, p, p)\n\n    def update(self, image, real):\n        for chunk in self.patches(image).split(32):\n            self.metric.update(chunk.float(), real=real)\n\n    def compute_and_reset(self):\n        value = self.metric.compute().item()\n        self.metric.reset()\n        return value\n',
    'nuse_e/models/__init__.py': '"""Model.\n\n    blocks     RepConv, ResBlock: khối của trunk\n    nuse_e     NuSEE = trunk theo stage + head hệ số + decoder PCA cố định + nhất quán dữ liệu\n    baselines  SPANV2, DISP, PLKSR-Tiny và bản cắm NuSE k=8 của thí nghiệm cũ\n    registry   build_model (tạo theo tên), make_nuse (NuSEE hoặc lớp con của thí nghiệm), load_run, budgets_of\n"""\nfrom .nuse_e import NuSEE\nfrom .registry import budgets_of, build_model, load_run, make_nuse\n\n__all__ = ["NuSEE", "budgets_of", "build_model", "load_run", "make_nuse"]\n',
    'nuse_e/models/baselines.py': '"""Baseline dựng lại từ thí nghiệm cũ: SPANV2, DISP, PLKSR-Tiny, và bản cắm NuSE k=8."""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom ..core.constants import S\nfrom ..operators import AvgPoolOp\n\nclass SPABV2(nn.Module):\n    """SPANV2 block as in the previous experiments: y = (x + f3) * Conv1x1(f3)."""\n\n    def __init__(self, in_channels, out_channels):\n        super().__init__()\n        self.c1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)\n        self.c2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)\n        self.c3 = nn.Conv2d(out_channels, out_channels, 3, padding=1)\n        self.gate = nn.Conv2d(out_channels, out_channels, 1)\n        self.act = nn.ReLU(inplace=True)\n        self.shortcut = nn.Identity() if in_channels == out_channels else nn.Conv2d(in_channels, out_channels, 1)\n\n    def forward(self, x):\n        f3 = self.c3(self.act(self.c2(self.act(self.c1(x)))))\n        return (self.shortcut(x) + f3) * self.gate(f3)\n\n\nclass SPANV2(nn.Module):\n    """SPANV2 (NTIRE 2026 ESR, XiaomiMM), same code as the previous SPANV2 runs."""\n\n    def __init__(self, channels=3, rank=None, width=32, num_blocks=5):\n        super().__init__()\n        near = channels * S * S\n        self.conv_near = nn.Conv2d(channels, near, 3, padding=1, groups=channels)\n        with torch.no_grad():\n            self.conv_near.weight.zero_()\n            self.conv_near.weight[:, 0, 1, 1] = 1\n            self.conv_near.bias.zero_()\n        self.blocks = nn.Sequential(*(SPABV2(channels if i == 0 else width, width) for i in range(num_blocks)))\n        fused = near + width\n        self.fuse_depthwise = nn.Conv2d(fused, fused, 3, padding=1, groups=fused)\n        self.fuse_pointwise = nn.Conv2d(fused, channels * S * S if rank is None else channels * rank, 1)\n        self.upsample = nn.PixelShuffle(S) if rank is None else nn.Identity()\n\n    def forward(self, lr):\n        x = torch.cat((self.conv_near(lr), self.blocks(lr)), dim=1)\n        return self.upsample(self.fuse_pointwise(self.fuse_depthwise(x)))\n\n\nclass DISPConv(nn.Module):\n    def __init__(self, c_in, c_out, s=1, bias=True):\n        super().__init__()\n        self.eval_conv = nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, stride=s, bias=bias)\n\n    def forward(self, x):\n        return self.eval_conv(x)\n\n\nclass REECB(nn.Module):\n    def __init__(self, in_channels, bias=False, extern_conv=0):\n        super().__init__()\n        self._extern_conv_block = nn.ModuleList(DISPConv(in_channels, in_channels, bias=bias)\n                                                for _ in range(extern_conv))\n        self.c1_r = DISPConv(in_channels, in_channels, bias=bias)\n        self.c2_r = DISPConv(in_channels, in_channels, bias=bias)\n        self.c3_r = DISPConv(in_channels, in_channels, bias=bias)\n        self.act1 = nn.SiLU(inplace=True)\n\n    def forward(self, x):\n        out2 = self.act1(self.c2_r(self.act1(self.c1_r(x))))\n        for extern in self._extern_conv_block:\n            out2 = self.act1(extern(out2))\n        return self.act1(self.c3_r(out2)) + x\n\n\nclass DISPNet(nn.Module):\n    def __init__(self, num_in_ch=3, feature_channels=32, upscale=4, out_channels=None):\n        super().__init__()\n        out_channels = num_in_ch * upscale ** 2 if out_channels is None else out_channels\n        self.conv_1 = DISPConv(num_in_ch, feature_channels, bias=True)\n        self.blocks = nn.Sequential(*[REECB(feature_channels, bias=True, extern_conv=1) for _ in range(4)])\n        self.conv_2 = nn.Conv2d(feature_channels, feature_channels, 1, 1, 0)\n        conv = nn.Conv2d(feature_channels, out_channels, 3, padding=1)\n        self.upsampler = (nn.Sequential(conv, nn.PixelShuffle(upscale))\n                          if out_channels == num_in_ch * upscale ** 2 else conv)\n\n    def forward(self, x):\n        feat = self.conv_1(x)\n        out = self.conv_2(self.blocks(feat)) + feat\n        return self.upsampler(out)\n\n\nclass DISPSR(nn.Module):\n    def __init__(self, channels=3, rank=None, width=32):\n        super().__init__()\n        self.body = DISPNet(channels, width, S, None if rank is None else channels * rank)\n\n    def forward(self, lr):\n        return self.body(lr)\n\n\nclass PLKConv2d(nn.Module):\n    def __init__(self, dim, kernel_size):\n        super().__init__()\n        self.conv = nn.Conv2d(dim, dim, kernel_size, 1, kernel_size // 2)\n        nn.init.trunc_normal_(self.conv.weight, std=0.02)\n        self.idx = dim\n\n    def forward(self, x):\n        x1, x2 = torch.split(x, [self.idx, x.size(1) - self.idx], dim=1)\n        return torch.cat([self.conv(x1), x2], dim=1)\n\n\nclass PLKBlock(nn.Module):\n    def __init__(self, dim, kernel_size, split_ratio):\n        super().__init__()\n        self.channe_mixer = nn.Sequential(nn.Conv2d(dim, dim * 2, 3, 1, 1), nn.GELU(),\n                                          nn.Conv2d(dim * 2, dim, 3, 1, 1))\n        self.lk = PLKConv2d(int(dim * split_ratio), kernel_size)\n        self.refine = nn.Conv2d(dim, dim, 1, 1, 0)\n\n    def forward(self, x):\n        return x + self.refine(self.lk(self.channe_mixer(x)))\n\n\nclass PLKSRNet(nn.Module):\n    def __init__(self, dim=64, n_blocks=12, kernel_size=13, split_ratio=0.25, out_channels=48):\n        super().__init__()\n        self.out_channels = out_channels\n        self.feats = nn.Sequential(nn.Conv2d(3, dim, 3, 1, 1),\n                                   *[PLKBlock(dim, kernel_size, split_ratio) for _ in range(n_blocks)],\n                                   nn.Conv2d(dim, out_channels, 3, 1, 1))\n        nn.init.trunc_normal_(self.feats[0].weight, std=0.02)\n        nn.init.trunc_normal_(self.feats[-1].weight, std=0.02)\n\n    def forward(self, x):\n        if self.out_channels == 3 * S * S:\n            return F.pixel_shuffle(self.feats(x) + x.repeat_interleave(S * S, dim=1), S)\n        return self.feats(x)\n\n\nclass PLKSRTiny(nn.Module):\n    def __init__(self, channels=3, rank=None, width=64):\n        super().__init__()\n        self.body = PLKSRNet(dim=width, out_channels=channels * S * S if rank is None else channels * rank)\n\n    def forward(self, lr):\n        return self.body(lr)\n\n\nBACKBONES = {"SPANV2": (SPANV2, 32), "DISP": (DISPSR, 32), "PLKSR-Tiny": (PLKSRTiny, 64)}\n\n\nclass ImageSR(nn.Module):\n    """Plain baseline trained with image MSE. State keys `predictor.*` match the previous checkpoints."""\n    kind = "image"\n\n    def __init__(self, name):\n        super().__init__()\n        cls, width = BACKBONES[name]\n        self.predictor = cls(3, None, width)\n\n    def forward(self, lr, k=None):\n        return self.predictor(lr)\n\n    def loss(self, lr, hr, alpha=None):\n        return F.mse_loss(self.predictor(lr), hr)\n\n\nclass LegacyNuSEPlugin(nn.Module):\n    """Previous `backbone + NuSE (k=8)` runs: coefficient head + fixed rank-8 decoder + A+y (average pool).\n\n    Only used to load and re-evaluate the previous checkpoints; buffer names match them.\n    """\n    kind = "legacy_plugin"\n\n    def __init__(self, name, state):\n        super().__init__()\n        cls, width = BACKBONES[name]\n        rank = state["decoder.components"].shape[-1]\n        self.predictor = cls(3, rank, width)\n        self.decoder = nn.Module()\n        for key in ("Q", "mean", "components", "eigenvalues", "synthesis_basis", "block_mean"):\n            self.decoder.register_buffer(key, state[f"decoder.{key}"].clone())\n        self.rank = rank\n        self.operator = AvgPoolOp()\n\n    def forward(self, lr, k=None):\n        coeffs = self.predictor(lr)\n        b, _, h, w = coeffs.shape\n        coeffs = coeffs.reshape(b, 3, self.rank, h, w)\n        blocks = torch.einsum("bckhw,cpk->bcphw", coeffs, self.decoder.synthesis_basis.to(coeffs.dtype))\n        blocks = blocks + self.decoder.block_mean[None, :, :, None, None].to(coeffs.dtype)\n        return self.operator.pinv(lr) + F.pixel_shuffle(blocks.reshape(b, 3 * 16, h, w), S)\n',
    'nuse_e/models/blocks.py': '"""Khối của trunk: RepConv và residual block."""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nclass RepConv(nn.Module):\n    """Conv3x3 trained as 3x3 + 1x1 + identity (structural re-parameterisation, RepVGG / ECB style);\n    `fuse()` folds the three branches into one Conv3x3, so inference cost equals a plain Conv3x3."""\n\n    def __init__(self, width):\n        super().__init__()\n        self.conv3 = nn.Conv2d(width, width, 3, padding=1)\n        self.conv1 = nn.Conv2d(width, width, 1)\n\n    def forward(self, x):\n        return self.conv3(x) + self.conv1(x) + x\n\n    @torch.no_grad()\n    def fuse(self):\n        width = self.conv3.in_channels\n        conv = nn.Conv2d(width, width, 3, padding=1).to(self.conv3.weight.device)\n        weight = self.conv3.weight + F.pad(self.conv1.weight, [1, 1, 1, 1])\n        weight[torch.arange(width), torch.arange(width), 1, 1] += 1.0\n        conv.weight.copy_(weight)\n        conv.bias.copy_(self.conv3.bias + self.conv1.bias)\n        return conv\n\n\nclass ResBlock(nn.Module):\n    """Generic residual block (ResNet / EDSR): Conv3x3 -> ReLU -> Conv3x3, identity skip.\n\n    kind = "plain": as above. "rep": both convs are RepConv while training (same inference block).\n    "lk7": Conv3x3 -> ReLU -> depthwise Conv7x7 -> Conv3x3 (larger receptive field for +50 params/channel).\n    """\n\n    def __init__(self, width, kind="plain"):\n        super().__init__()\n        conv = RepConv if kind == "rep" else (lambda w: nn.Conv2d(w, w, 3, padding=1))\n        self.conv1 = conv(width)\n        self.conv2 = conv(width)\n        self.dw = nn.Conv2d(width, width, 7, padding=3, groups=width) if kind == "lk7" else None\n\n    def forward(self, x):\n        f = F.relu(self.conv1(x))\n        if self.dw is not None:\n            f = self.dw(f)\n        return x + self.conv2(f)\n\n    def fuse(self):\n        for name in ("conv1", "conv2"):\n            if isinstance(getattr(self, name), RepConv):\n                setattr(self, name, getattr(self, name).fuse())\n\n    @staticmethod\n    def inference_params(width, kind):\n        return 2 * (9 * width * width + width) + (49 * width + width if kind == "lk7" else 0)\n',
    'nuse_e/models/nuse_e.py': '"""NuSE-E: trunk theo stage, head hệ số mỗi nhóm, decoder PCA cố định, nhất quán dữ liệu."""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom ..core.constants import S\nfrom .blocks import ResBlock\nfrom ..nullspace.architecture import loss_alpha\nfrom ..nullspace.basis import to_blocks\nfrom ..operators import make_operator\n\nclass GroupHead(nn.Module):\n    """c_G = sqrt(lambda_G) * (Conv1x1(f_g) [+ L_g(y)]); readout zero-initialised."""\n\n    def __init__(self, width, channels, size, std, linear):\n        super().__init__()\n        self.channels, self.size = channels, size\n        self.readout = nn.Conv2d(width, channels * size, 1)\n        nn.init.zeros_(self.readout.weight)\n        nn.init.zeros_(self.readout.bias)\n        self.linear = (nn.Conv2d(channels, channels * size, 3, padding=1, groups=channels,\n                                 padding_mode="replicate") if linear else None)\n        self.register_buffer("std", std.reshape(1, channels * size, 1, 1).float())\n\n    @torch.no_grad()\n    def set_linear(self, weights):\n        """weights (C, size, 10): 9 row-major taps + bias, in coefficient units."""\n        std = self.std.reshape(self.channels, self.size)\n        w = weights.to(std) / std.unsqueeze(-1)\n        self.linear.weight.copy_(w[..., :9].reshape(self.channels * self.size, 1, 3, 3))\n        self.linear.bias.copy_(w[..., 9].reshape(-1))\n\n    def forward(self, f, y):\n        out = self.readout(f)\n        if self.linear is not None:\n            out = out + self.linear(y)\n        return out * self.std\n\n\nclass FixedDecoder(nn.Module):\n    """z = mu + U_K c per channel: one grouped 1x1 conv + PixelShuffle. Buffers only."""\n\n    def __init__(self, mean_block, basis):\n        super().__init__()\n        self.channels = basis.shape[0]\n        self.register_buffer("mean_block", mean_block.float())          # (C,16)\n        self.register_buffer("basis", basis.float())                    # (C,16,d)\n\n    def forward(self, coeffs, k):\n        c = self.channels\n        weight = self.basis[..., :k].reshape(c * S * S, k, 1, 1).to(coeffs.dtype)\n        return F.pixel_shuffle(F.conv2d(coeffs, weight, self.mean_block.reshape(-1).to(coeffs.dtype), groups=c), S)\n\n    def targets(self, residual_blocks):\n        """c* = U^T (x_b - mu): (B,C,16,h,w) -> (B,C,d,h,w)."""\n        return torch.einsum("bcphw,cpk->bckhw", residual_blocks - self.mean_block[None, :, :, None, None],\n                            self.basis)\n\n\nclass NuSEE(nn.Module):\n    kind = "nuse_e"\n\n    def __init__(self, stats, arch, channels=3):\n        super().__init__()\n        width, kind = arch.get("width", 32), arch.get("block", "plain")\n        self.width, self.kind, self.alpha = width, kind, arch.get("alpha", "uniform")\n        self.method = stats["method"]\n        self.groups, self.blocks_per_stage = tuple(arch["groups"]), tuple(arch["blocks"])\n        self.budgets = list(arch["budgets"])\n        self.channels = channels\n        self.operator = make_operator(self.method)\n        mean, vec, lam = stats["mean"], stats["components"], stats["eigenvalues"]\n        if self.method == "average_pool":\n            q = stats["Q"]\n            basis = torch.einsum("pd,cdk->cpk", q, vec)                   # U = Q V\n            mean_block = torch.einsum("pd,cd->cp", q, mean)              # mu = Q m\n        else:\n            basis, mean_block = vec, mean\n        self.decoder = FixedDecoder(mean_block, basis)\n        self.stem = nn.Conv2d(channels, width, 3, padding=1)\n        self.stages = nn.ModuleList(nn.Sequential(*[ResBlock(width, kind) for _ in range(n)]) for n in arch["blocks"])\n        std = lam.clamp_min(1e-12).sqrt()\n        self.heads, start = nn.ModuleList(), 0\n        for size, lin in zip(arch["groups"], arch["linear"]):\n            head = GroupHead(width, channels, size, std[:, start:start + size], lin)\n            if lin:\n                head.set_linear(stats["wiener"][:, :, start:start + size].permute(0, 2, 1))\n            self.heads.append(head)\n            start += size\n\n    def coefficients(self, y, k):\n        g = self.budgets.index(k)\n        f, parts = self.stem(y), []\n        for j in range(g + 1):\n            f = self.stages[j](f)\n            parts.append(self.heads[j](f, y).reshape(y.shape[0], self.channels, self.groups[j], *y.shape[-2:]))\n        return torch.cat(parts, 2).flatten(1, 2)\n\n    def detail(self, coeffs, k):\n        return self.decoder(coeffs, k)\n\n    def consistency(self, z, y):\n        if self.method == "average_pool":\n            return z + self.operator.pinv(y)                             # Az = 0 (Thm. 2)\n        return z + self.operator.pinv(y - self.operator.forward(z))     # Thm. 6\n\n    def forward(self, y, k=None):\n        k = self.budgets[-1] if k is None else k\n        return self.consistency(self.detail(self.coefficients(y, k), k), y)\n\n    def fuse(self):\n        """Inference form: fold RepConv branches. Idempotent; a no-op for plain / lk7 blocks."""\n        for stage in self.stages:\n            for block in stage:\n                block.fuse()\n        return self\n\n    def params_for(self, k):\n        """Inference parameters (RepConv counted fused) of the model that serves budget k."""\n        g = self.budgets.index(k)\n        trunk = sum(ResBlock.inference_params(self.width, self.kind) for s in self.stages[:g + 1] for _ in s)\n        mods = [self.stem, *self.heads[:g + 1]]\n        return trunk + sum(p.numel() for m in mods for p in m.parameters())\n\n    def residual_blocks(self, hr):\n        if self.method == "average_pool":\n            return to_blocks(hr)                     # U^T removes the block mean (1^T U = 0)\n        return to_blocks(hr - self.operator.pinv(self.operator.forward(hr)))\n\n    def loss(self, lr, hr, alpha=None):\n        """Avg pool: weighted coefficient MSE (Thm. 4). Bicubic: sum of image MSE over budgets (Thm. 7)."""\n        kmax = self.budgets[-1]\n        alpha = loss_alpha(self.alpha if alpha is None else alpha, len(self.budgets))\n        chat = self.coefficients(lr, kmax).float()\n        with torch.autocast(lr.device.type, enabled=False):\n            return self._loss(chat, lr.float(), hr.float(), kmax, alpha)\n\n    def _loss(self, chat, lr, hr, kmax, alpha):\n        if self.method == "average_pool":\n            with torch.no_grad():\n                cstar = self.decoder.targets(self.residual_blocks(hr))\n            w = torch.zeros(kmax, device=lr.device)\n            for k, a in zip(self.budgets, alpha):\n                w[:k] += a\n            err = (chat.reshape_as(cstar) - cstar).square()\n            return (w[None, None, :, None, None] * err).sum() / hr.numel()\n        b = lr.shape[0]\n        chat = chat.reshape(b, self.channels, kmax, *lr.shape[-2:])\n        total = 0.0\n        for k, a in zip(self.budgets, alpha):\n            sr = self.consistency(self.detail(chat[:, :, :k].flatten(1, 2), k), lr)\n            total = total + a * F.mse_loss(sr, hr)\n        return total\n',
    'nuse_e/models/registry.py': '"""Tạo model theo tên và nạp checkpoint (của pipeline này hoặc của thí nghiệm cũ)."""\nimport importlib\n\nimport torch\n\nfrom .baselines import ImageSR, LegacyNuSEPlugin\nfrom .nuse_e import NuSEE\n\ndef make_nuse(stats, arch):\n    """NuSEE, or the subclass named by arch["impl"] ("module:Class", an experiment folder on sys.path)."""\n    if not arch.get("impl"):\n        return NuSEE(stats, arch)\n    module, cls = arch["impl"].split(":")\n    return getattr(importlib.import_module(module), cls)(stats, arch)\n\n\ndef build_model(name, method, stats=None, arch=None):\n    if name.startswith("NuSE-E"):                        # "NuSE-E" or "NuSE-E/<variant>"\n        return make_nuse(stats, arch)\n    return ImageSR(name)\n\n\ndef budgets_of(model):\n    return list(model.budgets) if isinstance(model, NuSEE) else [None]\n\n\ndef load_run(path, stats=None, arch=None):\n    """Load a checkpoint written by train_run.py or a previous-experiment best.pt."""\n    ck = torch.load(path, map_location="cpu", weights_only=False)\n    if "nuse_e" in ck:                                   # this notebook\n        info = ck["nuse_e"]\n        if info["model"].startswith("NuSE-E"):\n            model = make_nuse(ck["stats"], ck["arch"])\n            model.load_state_dict(ck["model"])\n            model.fuse()\n        else:\n            model = ImageSR(info["model"])\n            model.load_state_dict(ck["model"])\n        return model.eval(), ck\n    arch_old = ck["architecture"]                        # previous experiments\n    names = {"spanv2": "SPANV2", "disp": "DISP", "plksr_tiny": "PLKSR-Tiny"}\n    name = names[arch_old["backbone"]]\n    if arch_old.get("null_space", False):\n        model = LegacyNuSEPlugin(name, ck["model"])\n        model.load_state_dict(ck["model"])\n    else:\n        model = ImageSR(name)\n        model.load_state_dict(ck["model"])\n    return model.eval(), ck\n',
    'nuse_e/nullspace/__init__.py': '"""Lý thuyết PCA trong null space (plan/NuSE-E_thiet_ke_v2.md, mục 2 và 4).\n\n    basis         cơ sở Helmert Q của null space, tách ô 4×4, nhóm đối xứng D4\n    statistics    PCA làm đối xứng D4 + Wiener + R² k-fold trên DIV2K train -> stats.pt\n    architecture  quy tắc suy exit, nhóm hệ số, số block, nhánh tuyến tính từ phổ PCA\n"""\n',
    'nuse_e/nullspace/architecture.py': '"""Quy tắc suy kiến trúc từ phổ PCA: exit theo mức năng lượng (không cắt giữa cặp D4), block theo năng lượng phi\ntuyến, nhánh tuyến tính theo R².\n"""\nimport math\n\ndef allocate_blocks(energy, linear_r2, groups, total_blocks):\n    if total_blocks < len(groups):\n        raise ValueError("Need at least one block per budget")\n    start, nl = 0, []\n    for size in groups:\n        nl.append(sum(energy[i] * (1 - max(0.0, linear_r2[i])) for i in range(start, start + size)))\n        start += size\n    share = [total_blocks * e / sum(nl) for e in nl]\n    blocks = [max(1, math.floor(s)) for s in share]\n    while sum(blocks) < total_blocks:\n        rem = [s - b for s, b in zip(share, blocks)]\n        blocks[rem.index(max(rem))] += 1\n    while sum(blocks) > total_blocks:\n        over = [b - s if b > 1 else -1e9 for s, b in zip(share, blocks)]\n        blocks[over.index(max(over))] -= 1\n    return blocks, nl\n\n\ndef admissible_cuts(eigenvalues, rel_tol=1e-6):\n    d = len(eigenvalues)\n    cuts = [k for k in range(1, d) if eigenvalues[k - 1] - eigenvalues[k] > rel_tol * eigenvalues[k - 1]]\n    return cuts + [d]\n\n\ndef choose_budgets(eigenvalues, energy_targets=(0.5, 0.8, 0.9, 1.0), rel_tol=1e-6):\n    total = sum(eigenvalues)\n    cuts = admissible_cuts(eigenvalues, rel_tol)\n    budgets = []\n    for target in energy_targets:\n        k = next(k for k in cuts if sum(eigenvalues[:k]) / total >= target - 1e-12)\n        if k not in budgets:\n            budgets.append(k)\n    return budgets\n\n\ndef derive_architecture(eigenvalues, linear_r2, *, energy_targets=(0.5, 0.8, 0.9, 1.0), total_blocks=7,\n                        r2_min=0.05, rel_tol=1e-6, min_share=0.5):\n    budgets = choose_budgets(eigenvalues, energy_targets, rel_tol)\n\n    def nonlinear_energy(lo, hi):\n        return sum(eigenvalues[i] * (1 - max(0.0, linear_r2[i])) for i in range(lo, hi))\n\n    while len(budgets) > 1:\n        edges = [0] + budgets\n        e = [nonlinear_energy(edges[g], edges[g + 1]) for g in range(len(budgets))]\n        share = [total_blocks * v / sum(e) for v in e]\n        g = min(range(len(budgets) - 1), key=lambda i: share[i])\n        if share[g] >= min_share and len(budgets) <= total_blocks:\n            break\n        budgets.pop(g)\n    groups = [b - a for a, b in zip([0] + budgets[:-1], budgets)]\n    blocks, nonlinear = allocate_blocks(eigenvalues, linear_r2, groups, total_blocks)\n    linear, group_r2, start = [], [], 0\n    for size in groups:\n        lam = eigenvalues[start:start + size]\n        r2 = sum(l * max(0.0, r) for l, r in zip(lam, linear_r2[start:start + size])) / sum(lam)\n        group_r2.append(r2)\n        linear.append(r2 >= r2_min)\n        start += size\n    return {"budgets": budgets, "groups": groups, "blocks": blocks, "linear": linear,\n            "group_linear_r2": group_r2, "nonlinear_energy": nonlinear,\n            "rule": {"energy_targets": list(energy_targets), "total_blocks": total_blocks,\n                     "r2_min": r2_min, "rel_tol": rel_tol, "min_share": min_share}}\n\n\ndef architecture_from_stats(stats, width=32, block="plain", alpha="uniform", impl=None, options=None, **rule):\n    """Rule output plus the trunk hyper-parameters (width, block kind, loss weights) that the rule does not set.\n    `impl` ("module:Class", a NuSEE subclass from an experiment folder) and its `options` are stored as given."""\n    lam = stats["eigenvalues"].mean(0).tolist()\n    r2 = stats["r2"].mean(0).tolist()\n    arch = derive_architecture(lam, r2, **rule)\n    arch["method"] = stats["method"]\n    arch["retained_energy"] = [sum(lam[:k]) / sum(lam) for k in arch["budgets"]]\n    arch["width"], arch["block"], arch["alpha"] = width, block, alpha\n    if impl:\n        arch["impl"], arch["options"] = impl, dict(options or {})\n    return arch\n\n\ndef loss_alpha(spec, n_budgets):\n    """Thm. 4 holds for any alpha >= 0: \'uniform\' = every budget counts once; \'top\' = image MSE at K max only."""\n    if spec in (None, "uniform"):\n        return [1.0] * n_budgets\n    if spec == "top":\n        return [0.0] * (n_budgets - 1) + [1.0]\n    return list(spec)\n',
    'nuse_e/nullspace/basis.py': '"""Cơ sở Helmert của null space, tách ô 4×4, đối xứng D4, cửa sổ LR 3×3."""\nimport math\n\nimport torch\nfrom torch.nn import functional as F\n\nfrom ..core.constants import S\n\ndef helmert(scale=S, dtype=torch.float64):\n    """Q (q, q-1): Q^T Q = I, 1^T Q = 0."""\n    q = scale ** 2\n    basis = torch.zeros(q, q - 1, dtype=dtype)\n    for j in range(1, q):\n        norm = math.sqrt(j * (j + 1))\n        basis[:j, j - 1] = 1 / norm\n        basis[j, j - 1] = -j / norm\n    return basis\n\n\ndef to_blocks(x):\n    """(B,C,H,W) -> (B,C,16,H/4,W/4), pixel index p = dy*4 + dx."""\n    b, c, h, w = x.shape\n    return F.pixel_unshuffle(x, S).reshape(b, c, S * S, h // S, w // S)\n\n\ndef d4_permutations():\n    idx = torch.arange(S * S).reshape(S, S)\n    mats = []\n    for turns in range(4):\n        for flip in (False, True):\n            perm = torch.rot90(idx, turns, (0, 1))\n            perm = perm.flip(1) if flip else perm\n            p = torch.zeros(S * S, S * S, dtype=torch.float64)\n            p[torch.arange(S * S), perm.flatten()] = 1\n            mats.append(p)\n    return mats\n\n\ndef null_coordinates(hr, method, op, q):\n    """Per-channel null-space coordinates u of every block: (C, N, d)."""\n    if method == "average_pool":\n        blocks = to_blocks(hr)                                         # (B,C,16,h,w)\n        u = torch.einsum("bcphw,pd->cbhwd", blocks, q)\n    else:\n        r = hr - op.pinv(op.forward(hr))\n        u = to_blocks(r).permute(1, 0, 3, 4, 2)                        # (C,B,h,w,16)\n    return u.reshape(u.shape[0], -1, u.shape[-1])\n\n\ndef lr_windows(lr):\n    """Per-channel 3x3 replicate-padded LR windows + 1: (C, N, 10), row-major taps."""\n    b, c, h, w = lr.shape\n    x = F.pad(lr.reshape(b * c, 1, h, w), (1, 1, 1, 1), mode="replicate")\n    win = F.unfold(x, 3).reshape(b, c, 9, h * w).permute(1, 0, 3, 2).reshape(c, b * h * w, 9)\n    return torch.cat([win, torch.ones_like(win[..., :1])], -1)\n',
    'nuse_e/nullspace/statistics.py': '"""Thống kê từ DIV2K train: PCA làm đối xứng D4 trên phần dư null space, Wiener và R² k-fold (stats.pt)."""\nimport time\nfrom pathlib import Path\n\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom ..core.constants import S\nfrom ..data.cache import FlatImages\nfrom ..data.loaders import RamCropLoader, TrainCrops\nfrom .basis import d4_permutations, helmert, lr_windows, null_coordinates\nfrom ..operators import degrade, make_operator\n\n@torch.no_grad()\ndef fit_statistics(files, method, device, *, repeat=4, folds=5, batch=16, workers=2, seed=2026, log=print):\n    """Pass 1: D4-symmetrised PCA of the null-space residual. Pass 2: Wiener weights + k-fold R^2.\n\n    Training HR only (I8). FP64 accumulation. Returns a dict saved as stats.pt.\n    """\n    op = make_operator(method)\n    q = helmert().to(device)\n    d = 15 if method == "average_pool" else 16\n    frame = q if method == "average_pool" else torch.eye(16, dtype=torch.float64, device=device)\n    if isinstance(files, (str, Path)):                                  # flat uint8 cache (build_flat_cache)\n        images = FlatImages(files)\n        loader = RamCropLoader(images, batch, seed, 256, repeat, augment=False)\n        names = [e["name"] for e in images.index]\n    else:\n        data = TrainCrops(files, 256, repeat, augment=False, seed=seed)\n        loader = DataLoader(data, batch_size=batch, shuffle=False, num_workers=workers)\n        names = [Path(p).name for p in files]\n\n    def to_f64(hr):\n        return hr.to(device, torch.float64) / (255 if hr.dtype == torch.uint8 else 1)\n    n, total, outer = 0, None, None\n    t0 = time.time()\n    for step, (hr, _) in enumerate(loader):\n        hr = to_f64(hr)\n        u = null_coordinates(hr, method, op, q)                         # (C,N,d)\n        if total is None:\n            total = torch.zeros(u.shape[0], d, dtype=torch.float64, device=device)\n            outer = torch.zeros(u.shape[0], d, d, dtype=torch.float64, device=device)\n        total += u.sum(1)\n        outer += torch.einsum("cni,cnj->cij", u, u)\n        n += u.shape[1]\n        if step % 50 == 0:\n            log(f"[stats {method}] pass 1 batch {step + 1}/{len(loader)} ({time.time() - t0:.0f}s)")\n    mean = total / n\n    cov = outer / n - mean[:, :, None] * mean[:, None, :]\n    cov = (cov + cov.transpose(1, 2)) / 2\n    reps = [(frame.T @ p.to(device) @ frame) for p in d4_permutations()]\n    cov_sym = sum(r @ cov @ r.T for r in reps) / len(reps)\n    mean_sym = sum(mean @ r.T for r in reps) / len(reps)\n    lam, vec = torch.linalg.eigh(cov_sym)\n    lam, vec = lam.flip(-1).clamp_min(0), vec.flip(-1)                 # (C,d), (C,d,d) columns = v_i\n    invariance = max(((r @ cov_sym @ r.T) - cov_sym).abs().max().item() for r in reps) / cov_sym.abs().max().item()\n\n    xtx = torch.zeros(folds, 3, 10, 10, dtype=torch.float64, device=device)\n    xty = torch.zeros(folds, 3, 10, d, dtype=torch.float64, device=device)\n    ysum = torch.zeros(folds, 3, d, dtype=torch.float64, device=device)\n    ysq = torch.zeros(folds, 3, d, dtype=torch.float64, device=device)\n    cnt = torch.zeros(folds, dtype=torch.float64, device=device)\n    t0 = time.time()\n    for step, (hr, image_index) in enumerate(loader):\n        hr = to_f64(hr)\n        lr = degrade(hr, method)\n        u = null_coordinates(hr, method, op, q)\n        c = torch.einsum("cnd,cdk->cnk", u - mean_sym[:, None, :], vec)  # (C,N,d)\n        x = lr_windows(lr)                                              # (C,N,10)\n        per = (lr.shape[-2] * lr.shape[-1])\n        fold_of = (image_index % folds).to(device).repeat_interleave(per)\n        for f in range(folds):\n            sel = fold_of == f\n            if sel.any():\n                xs, cs = x[:, sel], c[:, sel]\n                xtx[f] += xs.transpose(1, 2) @ xs\n                xty[f] += xs.transpose(1, 2) @ cs\n                ysum[f] += cs.sum(1)\n                ysq[f] += cs.square().sum(1)\n                cnt[f] += sel.sum()\n        if step % 50 == 0:\n            log(f"[stats {method}] pass 2 batch {step + 1}/{len(loader)} ({time.time() - t0:.0f}s)")\n\n    def solve(a, b):\n        eps = 1e-8 * torch.diagonal(a, dim1=-2, dim2=-1).sum(-1)[..., None, None] / 10\n        return torch.linalg.solve(a + eps * torch.eye(10, dtype=a.dtype, device=a.device), b)\n\n    sse = torch.zeros(3, d, dtype=torch.float64, device=device)\n    sst = torch.zeros(3, d, dtype=torch.float64, device=device)\n    for f in range(folds):\n        keep = [g for g in range(folds) if g != f]\n        w = solve(xtx[keep].sum(0), xty[keep].sum(0))                  # (C,10,d)\n        sse += ysq[f] - 2 * torch.einsum("cid,cid->cd", w, xty[f]) + torch.einsum("cid,cij,cjd->cd", w, xtx[f], w)\n        sst += ysq[f] - ysum[f].square() / cnt[f].clamp_min(1)\n    r2 = 1 - sse / sst.clamp_min(1e-30)\n    wiener = solve(xtx.sum(0), xty.sum(0))                              # (C,10,d)\n    return {"method": method, "scale": S, "d": d, "mean": mean_sym.cpu(), "components": vec.cpu(),\n            "eigenvalues": lam.cpu(), "wiener": wiener.cpu(), "r2": r2.cpu(), "blocks_per_channel": n,\n            "Q": helmert(), "d4_invariance": invariance, "n_train_images": len(names),\n            "files": names, "torch_version": torch.__version__}\n',
    'nuse_e/operators.py': '"""Toán tử đo A (tạo LR từ HR) và giả nghịch đảo A⁺.\n\naverage_pool: trung bình ô 4×4, A⁺ = lặp điểm. bicubic: bicubic ×4 có antialias, A⁺ dạng băng (Định lý 5–6)."""\nimport torch\nfrom torch.nn import functional as F\n\nfrom .core.constants import S\n\ndef avg_down(x):\n    return F.avg_pool2d(x, S)\n\n\ndef bicubic_down(x):\n    """PyTorch antialiased bicubic x1/4 (a = -0.5 inside the image, same as MATLAB imresize)."""\n    return F.interpolate(x, scale_factor=1 / S, mode="bicubic", align_corners=False, antialias=True)\n\n\ndef degrade(x, method):\n    return avg_down(x) if method == "average_pool" else bicubic_down(x)\n\n\nclass AvgPoolOp:\n    name = "average_pool"\n\n    def forward(self, x):\n        return avg_down(x)\n\n    def pinv(self, y):\n        return y.repeat_interleave(S, -2).repeat_interleave(S, -1)\n\n\nclass BicubicOp:\n    """B(x) = M_h x M_w^T (Thm. 5); exact separable pseudo-inverse B+ = M_h^+ (.) M_w^+T.\n\n    M is extracted from bicubic_down itself (impulse responses), so B is exactly the\n    function that generates the LR data. Matrices are cached per (H, W, device, dtype).\n    """\n    name = "bicubic"\n\n    def __init__(self):\n        self.cpu_cache = {}            # (H, W) -> FP64 matrices on CPU; one entry per image size\n\n    @staticmethod\n    def matrix(n):\n        eye = torch.eye(n, dtype=torch.float64).reshape(n, 1, 1, n).expand(n, 1, S, n)\n        out = bicubic_down(eye)                                        # (n,1,1,n/S)\n        return out.reshape(n, n // S).T.contiguous()                   # (n/S, n)\n\n    def mats(self, h, w, device, dtype):\n        if (h, w) not in self.cpu_cache:\n            mh, mw = self.matrix(h), self.matrix(w)\n            self.cpu_cache[(h, w)] = tuple(t.float() for t in (mh, mw, torch.linalg.pinv(mh), torch.linalg.pinv(mw)))\n        key = (h, w, str(device), dtype)\n        if getattr(self, "_last_key", None) != key:\n            self._last = tuple(t.to(device=device, dtype=dtype) for t in self.cpu_cache[(h, w)])\n            self._last_key = key\n        return self._last\n\n    def forward(self, x):\n        return bicubic_down(x)\n\n    def forward_matrix(self, x):\n        mh, mw, _, _ = self.mats(x.shape[-2], x.shape[-1], x.device, x.dtype)\n        return mh @ x @ mw.T\n\n    def pinv(self, y):\n        _, _, ph, pw = self.mats(y.shape[-2] * S, y.shape[-1] * S, y.device, y.dtype)\n        return ph @ y @ pw.T\n\n\ndef make_operator(method):\n    return AvgPoolOp() if method == "average_pool" else BicubicOp()\n',
    'nuse_e/pipeline/__init__.py': '"""NuSE-E: toàn bộ quy trình thí nghiệm, chạy từ dòng lệnh (thay cho notebook).\n\nMỗi bước đọc đầu vào và ghi kết quả ra thư mục output, nên chạy lại một bước bất kỳ không cần chạy lại các bước\ntrước (khác notebook, nơi khởi động lại kernel làm mất mọi biến). Thứ tự các bước:\n\n    setup     tìm dữ liệu, tạo cache ảnh, kiểm checkpoint cũ, (tuỳ chọn) chép kết quả phiên trước\n    stats     thống kê PCA null space + suy kiến trúc cho mọi biến thể\n    test      test chấp nhận (UᵀU = I, A(SR) = y, co giãn theo K, ...)\n    screen    sàng tham số và latency trước khi train\n    train     train mọi run còn thiếu (một tiến trình mỗi GPU), rồi pha B trên LR bicubic\n    curves    hình đường học\n    select    chọn biến thể tốt nhất mỗi loại LR\n    evaluate  PSNR, SSIM, kênh Y, LPIPS, FID trên DIV2K valid (và benchmark nếu có)\n    cost      tham số, GFLOPs, latency\n    report    bảng tổng hợp, kiểm ràng buộc, report.md\n    figures   hình so sánh\n    all       chạy lần lượt mọi bước trên\n\nMỗi bước nằm trong một module của package này: prepare (setup, stats), checks (test, screen),\ntraining (train), selection (select), evaluation (evaluate, cost), report (report),\nfigures (curves, figures). Bối cảnh dùng chung ở context.\n\nVí dụ: python -m nuse_e.pipeline setup --train-hr <DIV2K_train_HR> --val-hr <DIV2K_valid_HR>\n       python -m nuse_e.pipeline all\nXem nuse_e/README.md.\n"""\n',
    'nuse_e/pipeline/__main__.py': '"""Dòng lệnh: python -m nuse_e.pipeline <bước> [<bước> ...] [tuỳ chọn]. Xem nuse_e/pipeline/__init__.py."""\nimport argparse\nimport time\n\nfrom . import __doc__ as PIPELINE_DOC\nfrom .checks import step_screen, step_test\nfrom .context import Ctx\nfrom .evaluation import step_cost, step_evaluate\nfrom .figures import step_curves, step_figures\nfrom .prepare import step_setup, step_stats\nfrom .report import step_report\nfrom .selection import step_select\nfrom .training import step_train\n\nSTEPS = {"setup": step_setup, "stats": step_stats, "test": step_test, "screen": step_screen, "train": step_train,\n         "curves": step_curves, "select": step_select, "evaluate": step_evaluate, "cost": step_cost,\n         "report": step_report, "figures": step_figures}\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=PIPELINE_DOC, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("steps", nargs="+", choices=[*STEPS, "all"], help="một hoặc nhiều bước, chạy theo thứ tự đã ghi")\n    parser.add_argument("--out", default="outputs/nuse_e", help="thư mục output (mặc định outputs/nuse_e)")\n    parser.add_argument("--data-root", help="thư mục để tự tìm DIV2K_train_HR / DIV2K_valid_HR")\n    parser.add_argument("--train-hr", help="thư mục DIV2K_train_HR")\n    parser.add_argument("--val-hr", help="thư mục DIV2K_valid_HR")\n    parser.add_argument("--old-runs", help="thư mục chứa checkpoint cũ (spanv2_x4/best.pt, disp_nuse_x4_k8/best.pt, ...)")\n    parser.add_argument("--benchmarks", help="thư mục chứa Set5, Set14, B100, Urban100, Manga109 (ảnh HR)")\n    parser.add_argument("--cache", help="thư mục cache ảnh (mặc định <out>/cache)")\n    parser.add_argument("--resume-from", help="bước setup: chép stats/ và runs/ từ output phiên trước")\n    parser.add_argument("--experiment", help="thư mục thí nghiệm có experiment.py (biến thể, thiết lập riêng; "\n                                             "xem experiments/README.md). Được nhớ trong <out>/paths.json")\n    parser.add_argument("--variants", help="chỉ dùng các biến thể này, ví dụ e80,d8 (mặc định: mọi biến thể trong config.py)")\n    parser.add_argument("--device", help="cuda hoặc cpu (mặc định: cuda nếu có)")\n    parser.add_argument("--gpus", help="bước train: danh sách GPU, ví dụ 0,1 (mỗi GPU một run)")\n    parser.add_argument("--deadline", type=float, help="bước train: mốc thời gian Unix phải dừng; run đang chạy lưu "\n                                                       "last.pt rồi dừng, không nhận run mới (chạy lại để tiếp tục)")\n    parser.add_argument("--smoke", action="store_true", help="chạy thử cực ngắn để kiểm code")\n    args = parser.parse_args(argv)\n    ctx = Ctx(args)\n    steps = list(STEPS) if "all" in args.steps else args.steps\n    for name in steps:\n        print(f"\\n==================== {name} ====================", flush=True)\n        t0 = time.time()\n        STEPS[name](ctx)\n        print(f"[{name}] xong sau {time.time() - t0:.0f}s", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n',
    'nuse_e/pipeline/checks.py': '"""Bước test (test chấp nhận) và screen (sàng tham số, latency trước khi train)."""\nimport statistics\nimport sys\n\nimport torch\n\nfrom ..core.io import write_csv, write_json\nfrom ..data.images import ValImages, as_float\nfrom ..evaluation.cost import time_cuda\nfrom ..models.blocks import ResBlock\nfrom ..models.registry import load_run, make_nuse\nfrom ..nullspace.basis import helmert, lr_windows\nfrom ..operators import degrade, make_operator\n\ndef step_test(ctx):\n    """Test chấp nhận: phải pass hết trước khi train."""\n    c = ctx.c\n    results = []\n\n    def check(name, ok, value=""):\n        print(f"{\'PASS\' if ok else \'FAIL\'}  {name}  {value}")\n        results.append(bool(ok))\n\n    q = helmert()\n    check("T2 QᵀQ = I, 1ᵀQ = 0", (q.T @ q - torch.eye(15, dtype=q.dtype)).abs().max() < 1e-12 and q.sum(0).abs().max() < 1e-12)\n    val_files = ctx.data()["val_files"]\n    for method in c.METHODS:\n        stats = torch.load(ctx.stats_dir / method / "stats.pt", map_location="cpu", weights_only=False)\n        op = make_operator(method)\n        hr = as_float(ValImages(val_files)[0][0], "cpu").double()\n        lr_data = degrade(hr, method)\n        if method == "bicubic":\n            err = (op.forward_matrix(hr) - lr_data).abs().max()\n            check(f"T1 [{method}] B (ma trận) = hàm tạo LR", err < 1e-6, f"{err:.1e}")\n        if "d4_invariance" in stats:\n            check(f"T3 [{method}] bất biến D4", stats["d4_invariance"] < 1e-12, f"{stats[\'d4_invariance\']:.1e}")\n        y = degrade(torch.rand(1, 3, 128, 160), method)\n        for name in c.NUSE:\n            arch = ctx.arch(method, name)\n            tag = f"[{method} {name}]"\n            model = make_nuse(stats, arch).eval()\n            u = model.decoder.basis.double()\n            check(f"T4 {tag} UᵀU = I", (torch.einsum("cpk,cpj->ckj", u, u) - torch.eye(u.shape[-1])).abs().max() < 1e-5)\n            worst = 0.0\n            for seed in range(3):\n                torch.manual_seed(seed)\n                net = make_nuse(stats, arch).eval()\n                with torch.no_grad():\n                    for p in net.parameters():\n                        p.normal_(0, 1)\n                    for k in net.budgets:\n                        z = net.detail(net.coefficients(y, k), k)\n                        worst = max(worst, ((op.forward(net(y, k)) - y).abs().max() / z.abs().max().clamp_min(1)).item())\n            check(f"T5 {tag} A(SR)=y, trọng số ngẫu nhiên, sai số / max|z|", worst < 1e-6, f"{worst:.1e}")\n            with torch.no_grad():\n                k1 = model.budgets[min(1, len(model.budgets) - 1)]\n                before = model(y, k1).clone()\n                g1 = model.budgets.index(k1)\n                later = [*model.stages[g1 + 1:], *model.heads[g1 + 1:], *getattr(model, "mixers", [])[g1 + 1:]]\n                for mod in later:\n                    for p in mod.parameters():\n                        p.add_(torch.randn_like(p))\n                check(f"T7 {tag} output ở K={k1} không phụ thuộc stage sau", torch.equal(before, model(y, k1)))\n            model = make_nuse(stats, arch)\n            W, blk = arch["width"], ResBlock.inference_params(arch["width"], arch["block"])\n            extra = getattr(model, "extra_inference_params", lambda g: 0)    # module riêng của lớp con thí nghiệm\n            formula = [28 * W + blk * sum(arch["blocks"][:g + 1]) + sum((3 * W + 3) * n + 30 * n * l for n, l in\n                       zip(arch["groups"][:g + 1], arch["linear"][:g + 1])) + extra(g) for g in range(len(arch["budgets"]))]\n            params = [model.params_for(k) for k in model.budgets]\n            check(f"T8 {tag} số tham số suy luận theo công thức", params == formula, params)\n            check(f"T9 {tag} mỗi ngân sách thêm ≥ 1 block", min([params[0]] + [b - a for a, b in zip(params, params[1:])]) >= blk)\n            if arch["block"] not in ("plain", "lk7"):                      # block gộp nhánh: rep, acb, ecb, ...\n                torch.manual_seed(0)\n                net = make_nuse(stats, arch).eval()\n                with torch.no_grad():\n                    for p in net.parameters():\n                        p.normal_(0, 0.3)\n                    outs = [net(y, k).clone() for k in net.budgets]\n                    net.fuse()\n                    fused = [net(y, k) for k in net.budgets]\n                    n_params = sum(p.numel() for p in net.parameters())\n                err = max(((a - b).abs().max() / a.abs().max()).item() for a, b in zip(outs, fused))\n                check(f"T11 {tag} gộp nhánh ({arch[\'block\']}): output bằng nhau, tham số = công thức",\n                      err < 1e-5 and n_params == params[-1], f"{err:.1e}, {n_params}")\n            if arch["linear"][0]:\n                lr = degrade(torch.rand(2, 3, 64, 64, dtype=torch.float64), method)\n                g0 = arch["groups"][0]\n                with torch.no_grad():\n                    got = model.eval().coefficients(lr.float(), model.budgets[0]).double().reshape(2, 3, g0, 16, 16)\n                x = lr_windows(lr)\n                want = torch.einsum("cni,cik->cnk", x, stats["wiener"][:, :, :g0]).reshape(3, 2, 16, 16, g0).permute(1, 0, 4, 2, 3)\n                rel = ((got - want).abs().max() / want.abs().max()).item()\n                check(f"T10 {tag} lúc khởi tạo head = Wiener", rel < 1e-5, f"{rel:.1e}")\n    if not all(results):\n        sys.exit("Có test FAIL: dừng, không train.")\n    print(f"Pass hết {len(results)} test.")\n\n\n@torch.no_grad()\ndef quick_ms(fn, warmup=20, iters=100):\n    t, _ = time_cuda({"x": fn}, "fp32", warmup, iters, 2)\n    return statistics.median(t["x"])\n\n\ndef step_screen(ctx):\n    """Tham số và latency (K lớn nhất, LR 256², fp32) của mọi biến thể; vượt trần thì không train."""\n    c = ctx.c\n    cap_ms, cap_src = c.LATENCY_CAP_MS, "số cũ (T4)"\n    cuda = ctx.device.type == "cuda"\n    if cuda:\n        torch.backends.cudnn.benchmark = True\n        x256 = torch.rand(1, 3, 256, 256, device=ctx.device)\n        if c.LATENCY_REF in ctx.old():\n            ref_model = load_run(ctx.old()[c.LATENCY_REF])[0].to(ctx.device).eval()\n            cap_ms, cap_src = quick_ms(lambda: ref_model(x256)), f"đo lại trên {torch.cuda.get_device_name(0)}"\n            del ref_model\n    print(f"Trần: tham số <= {c.PARAM_CAP:,} | latency @256² fp32 <= {cap_ms:.2f} ms ({cap_src})"\n          + ("" if cuda else " | không có GPU: bỏ qua kiểm latency"))\n    method = "average_pool" if "average_pool" in c.METHODS else c.METHODS[0]   # trunk giống nhau giữa hai loại LR\n    stats = torch.load(ctx.stats_dir / method / "stats.pt", map_location="cpu", weights_only=False)\n    rows = []\n    for name in c.NUSE:\n        arch = ctx.arch(method, name)\n        model = make_nuse(stats, arch).fuse().to(ctx.device).eval()\n        kmax = model.budgets[-1]\n        params = model.params_for(kmax)\n        ms = quick_ms(lambda m=model: m(x256, kmax)) if cuda else None\n        ok = params <= c.PARAM_CAP and (ms is None or ms <= cap_ms)\n        rows.append({"model": name, "params_kmax": params, "ms_256_fp32": ms, "admissible": ok})\n        print(f"{\'OK  \' if ok else \'LOẠI\'} {name:14s} W={arch[\'width\']} {arch[\'block\']:5s} N={sum(arch[\'blocks\'])} "\n              f"K={arch[\'budgets\']} | tham số K max {params:,} | {(\'%.2f ms\' % ms) if ms else \'—\'}")\n        del model\n    write_csv(ctx.compare / "screen.csv", rows)\n    write_json(ctx.compare / "screen.json", {"cap_ms": cap_ms, "cap_src": cap_src, "device": str(ctx.device), "rows": rows})\n    if not next(r for r in rows if r["model"] == "NuSE-E")["admissible"]:\n        sys.exit("Bản tham chiếu v2 phải qua sàng; nếu không thì trần đo sai.")\n',
    'nuse_e/pipeline/context.py': '"""Bối cảnh dùng chung của mọi bước: đường dẫn, thiết lập, trạng thái đã ghi trên đĩa."""\nimport importlib.util\nimport json\nimport os\nimport sys\nimport time\nfrom concurrent.futures import ProcessPoolExecutor\nfrom pathlib import Path\n\nimport torch\n\nfrom .. import config as CONFIG\nfrom ..core.constants import IMAGE_EXT\nfrom ..data.cache import build_flat_cache\nfrom ..data.images import image_files, to_bmp\nfrom ..models.registry import budgets_of, load_run\n\nPACKAGE_PARENT = Path(__file__).resolve().parents[2]   # thư mục chứa package nuse_e (gốc repo)\nPATH_KEYS = ("data_root", "train_hr", "val_hr", "old_runs", "benchmarks", "cache", "experiment")\n\n\nclass Ctx:\n    """Đường dẫn, thiết lập và trạng thái đã ghi trên đĩa của một thư mục output."""\n\n    def __init__(self, args):\n        self.args = args\n        self.out = Path(args.out).resolve()\n        self.stats_dir, self.runs, self.compare = self.out / "stats", self.out / "runs", self.out / "comparison"\n        self.fig = self.compare / "figures"\n        for path in (self.out, self.stats_dir, self.runs, self.compare, self.fig):\n            path.mkdir(parents=True, exist_ok=True)\n        self.paths = self._paths(args)\n        self.c = CONFIG.load(args.smoke, args.variants.split(",") if args.variants else None,\n                             load_experiment(self.paths.get("experiment")))\n        self.cache = Path(self.paths.get("cache") or self.out / "cache")\n        self.device = torch.device(args.device or ("cuda" if torch.cuda.is_available() else "cpu"))\n        self._data, self._old = None, None\n\n    # ---- đường dẫn dữ liệu: nhớ trong <out>/paths.json để các bước sau khỏi truyền lại\n    def _paths(self, args):\n        file = self.out / "paths.json"\n        paths = json.loads(file.read_text()) if file.exists() else {}\n        for key in PATH_KEYS:\n            value = getattr(args, key, None)\n            if value:\n                paths[key] = str(Path(value).expanduser().resolve())\n        file.write_text(json.dumps(paths, indent=2, ensure_ascii=False))\n        return paths\n\n    def find_dir(self, words, minimum, exclude=("_lr", "lr_", "x2", "x3", "x4", "x8"), root=None):\n        root = Path(root or self.paths.get("data_root") or "")\n        if not root or not root.is_dir():\n            return None\n        best = None\n        for folder_root, dirs, _ in os.walk(root, followlinks=True):\n            for name in dirs:\n                folder = Path(folder_root) / name\n                text = str(folder).lower()\n                if any(w in text for w in exclude) or not all(w in text for w in words):\n                    continue\n                n = image_count(folder)\n                if n >= minimum and (best is None or n > best[1]):\n                    best = (folder, n)\n        return best[0] if best else None\n\n    # ---- dữ liệu: cache uint8 phẳng cho train, BMP cho valid (tạo một lần, dùng lại)\n    def data(self):\n        if self._data:\n            return self._data\n        c = self.c\n        train_src = self.paths.get("train_hr") or self.find_dir(("div2k", "train", "hr"), 6 if c.SMOKE else 500)\n        val_src = self.paths.get("val_hr") or self.find_dir(("div2k", "valid", "hr"), 2 if c.SMOKE else 50)\n        if not train_src or not val_src:\n            sys.exit("Không tìm thấy DIV2K train/valid HR. Chạy: python -m nuse_e.pipeline setup "\n                     "--train-hr <DIV2K_train_HR> --val-hr <DIV2K_valid_HR> (hoặc --data-root <thư mục chứa DIV2K>)")\n        t0 = time.time()\n        workers = max(1, os.cpu_count() or 1)\n        train_files = image_files(train_src)\n        train_cache = build_flat_cache(train_files, self.cache / "div2k_train_u8.bin", workers=workers)\n        folder = self.cache / "div2k_valid_bmp"\n        folder.mkdir(parents=True, exist_ok=True)\n        with ProcessPoolExecutor(max_workers=workers) as pool:\n            val_files = [Path(p) for p in pool.map(to_bmp, [(str(p), str(folder)) for p in image_files(val_src)],\n                                                   chunksize=8)]\n        benchmarks = {}\n        bench_root = self.paths.get("benchmarks") or self.paths.get("data_root")\n        for name in c.BENCHMARK_NAMES:\n            found = self.find_dir((name.lower(),), 5, root=bench_root,\n                                  exclude=("_lr", "lr_", "lrbi", "x2", "x3", "x4", "x8", "bicubic"))\n            if found:\n                benchmarks[name] = image_files(found)\n        print(f"Train HR: {train_src} ({len(train_files)} ảnh, cache {train_cache.stat().st_size / 1e9:.2f} GB) | "\n              f"Valid HR: {val_src} ({len(val_files)} ảnh) | benchmark: {({k: len(v) for k, v in benchmarks.items()} or \'không có\')} "\n              f"| {time.time() - t0:.0f}s")\n        self._data = dict(train_files=train_files, train_cache=train_cache, val_files=val_files, benchmarks=benchmarks)\n        return self._data\n\n    # ---- checkpoint của thí nghiệm cũ (baseline, bản cắm NuSE k=8)\n    def old(self):\n        if self._old is None:\n            root = Path(self.paths["old_runs"]) if self.paths.get("old_runs") else None\n            self._old = {}\n            for label, folder in self.c.OLD_NAMES.items():\n                hits = sorted(root.rglob(f"{folder}/best.pt")) if root and root.is_dir() else []\n                if hits:\n                    self._old[label] = hits[0]\n        return self._old\n\n    # ---- tên, thư mục, kiến trúc\n    def design_of(self, model):\n        return {**self.c.DESIGN, **self.c.VARIANTS[model.split("/")[1]]} if "/" in model else dict(self.c.DESIGN)\n\n    def run_dir(self, method, model):\n        return self.runs / method / slug(model)\n\n    def arch_path(self, method, model):\n        return self.stats_dir / method / ("arch.json" if model == "NuSE-E" else f"arch_{model.split(\'/\')[1]}.json")\n\n    def arch(self, method, model):\n        path = self.arch_path(method, model)\n        if not path.exists():\n            sys.exit(f"Chưa có {path}: chạy bước stats trước.")\n        return json.loads(path.read_text())\n\n    def is_done(self, method, model):\n        return (self.run_dir(method, model) / "DONE").exists()\n\n    def uses_old(self, method, model):\n        return self.c.OLD_RUNS == "reuse" and method == "average_pool" and model in self.old()\n\n    # ---- trạng thái ghi bởi các bước trước\n    def screen(self):\n        path = self.compare / "screen.json"\n        if not path.exists():\n            sys.exit("Chưa có comparison/screen.json: chạy bước screen trước.")\n        return json.loads(path.read_text())\n\n    def admissible(self, method):\n        return [r["model"] for r in self.screen()["rows"] if r["admissible"]]\n\n    def selected(self):\n        path = self.compare / "selected.json"\n        if not path.exists():\n            sys.exit("Chưa có comparison/selected.json: chạy bước select trước.")\n        return json.loads(path.read_text())\n\n    def full(self, method):\n        """Model được đánh giá đầy đủ (LPIPS, FID, latency mọi K): baseline, bản v2, bản được chọn."""\n        return {*self.c.BASELINES, "NuSE-E", self.selected().get(method)} - {None}\n\n    def report(self, method):\n        path = self.compare / method / "report.json"\n        if not path.exists():\n            sys.exit(f"Chưa có {path}: chạy bước evaluate trước.")\n        return {(r["set"], r["model"]): {k: v for k, v in r.items() if k not in ("set", "model")}\n                for r in json.loads(path.read_text())}\n\n    def per_image(self, method):\n        path = self.compare / method / "per_image.json"\n        rows = json.loads(path.read_text()) if path.exists() else []\n        out = {}\n        for r in rows:\n            out.setdefault((r["set"], r["model"]), []).append({k: v for k, v in r.items() if k not in ("set", "model")})\n        return out\n\n    def load_list(self, name):\n        path = self.compare / name\n        return json.loads(path.read_text()) if path.exists() else []\n\n    def models_eval(self, method):\n        """{nhãn: (model, k)} của mọi model có checkpoint cho loại LR này."""\n        entries = {}\n        for model_name in self.c.MODELS:\n            path = self.old()[model_name] if self.uses_old(method, model_name) else self.run_dir(method, model_name) / "best.pt"\n            if not Path(path).exists() or (model_name.startswith("NuSE-E") and not self.is_done(method, model_name)):\n                print(f"không có {model_name} / {method} -> bỏ qua")\n                continue\n            model, _ = load_run(path)\n            model = model.to(self.device).eval()\n            for k in budgets_of(model):\n                entries[model_name if k is None else f"{model_name} K={k}"] = (model, k)\n        if method == "average_pool" and self.c.SHOW_OLD_PLUGIN:\n            for label, path in self.old().items():\n                if "NuSE k=8" in label:\n                    entries[label] = (load_run(path)[0].to(self.device).eval(), None)\n        return entries\n\n\ndef load_experiment(folder):\n    """Thiết lập viết hoa trong <folder>/experiment.py; thư mục được thêm vào sys.path để nạp model của thí nghiệm."""\n    if not folder:\n        return None\n    if folder not in sys.path:\n        sys.path.insert(0, folder)\n    spec = importlib.util.spec_from_file_location("nuse_e_experiment", Path(folder) / "experiment.py")\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    print("Thí nghiệm:", folder, "|", (module.__doc__ or "").strip().splitlines()[0] if module.__doc__ else "")\n    return {k: getattr(module, k) for k in dir(module) if k.isupper()}\n\n\ndef slug(model):\n    return {"SPANV2": "spanv2", "DISP": "disp", "PLKSR-Tiny": "plksr_tiny"}.get(model, "nuse_e" + model[6:].replace("/", "_"))\n\n\ndef name_of(label):\n    return label.split(" K=")[0]\n\n\ndef image_count(folder):\n    try:\n        return sum(1 for p in Path(folder).iterdir() if p.suffix.lower() in IMAGE_EXT)\n    except OSError:\n        return 0\n\n\ndef md_table(header, rows):\n    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]\n    return "\\n".join(lines + ["| " + " | ".join(map(str, r)) + " |" for r in rows])\n\n\ndef fmt(v, d=4):\n    return "—" if v is None or v != v else ("∞" if v == float("inf") else f"{v:.{d}f}")\n',
    'nuse_e/pipeline/evaluation.py': '"""Bước evaluate (chất lượng ảnh) và cost (tham số, GFLOPs, latency)."""\nimport copy\nimport json\nimport statistics\nimport time\nfrom pathlib import Path\n\nimport torch\n\nfrom ..core.io import write_csv, write_json\nfrom ..data.images import ValImages\nfrom ..evaluation.cost import count_params, gflops, stage_functions, time_cpu, time_cuda\nfrom ..evaluation.evaluate import evaluate, evaluate_bicubic_upsampling\nfrom ..evaluation.perceptual import LPIPSMetric, PatchFID\nfrom .context import name_of\n\ndef metric_objects(ctx):\n    lpips = fid_cls = None\n    if ctx.c.USE_LPIPS:\n        try:\n            lpips = LPIPSMetric().to(ctx.device)\n        except ImportError:\n            print("Thiếu gói lpips: bỏ LPIPS (pip install lpips)")\n    if ctx.c.USE_FID:\n        try:\n            PatchFID(ctx.device)\n            fid_cls = PatchFID\n        except (ImportError, ModuleNotFoundError) as error:\n            print(f"Thiếu gói cho FID ({error}): bỏ FID (pip install torchmetrics torch-fidelity)")\n    return lpips, fid_cls\n\n\ndef step_evaluate(ctx):\n    """PSNR/SSIM RGB (SR thô, shave 0), kênh Y (uint8, shave 4), LPIPS, FID patch 256, max|A(SR) - y|."""\n    c, data = ctx.c, ctx.data()\n    full = {m: ctx.full(m) for m in c.METHODS}\n    lpips, fid_cls = metric_objects(ctx)\n    val_images = [img for img in ValImages(data["val_files"])]\n    for method in c.METHODS:\n        models = ctx.models_eval(method)\n        report, per_image = {}, {}\n        sets = {"DIV2K-valid": val_images, **{n: [img for img in ValImages(f)] for n, f in data["benchmarks"].items()}}\n        for set_name, images in sets.items():\n            fid = fid_cls(ctx.device) if fid_cls and len(images) >= 50 else None\n            summary, _ = evaluate_bicubic_upsampling(images, method, ctx.device, lpips=lpips, fid=fid, fid_real=True)\n            report[(set_name, "Bicubic")] = summary\n            for label, (model, k) in models.items():\n                t0 = time.time()\n                is_full = name_of(label) in full[method] or "(cũ)" in label\n                save = (ctx.out / "sr_images" / method / label.replace(" ", "_")\n                        if c.SAVE_SR_IMAGES and set_name == "DIV2K-valid" else None)\n                summary, rows = evaluate(model, images, method, ctx.device, k=k, lpips=lpips if is_full else None,\n                                           fid=fid if is_full else None, save_dir=save)\n                report[(set_name, label)] = summary\n                per_image[(set_name, label)] = rows\n                print(f"[{method}] {set_name:12s} {label:30s} PSNR {summary[\'rgb_psnr\']:.4f}  SSIM {summary[\'rgb_ssim\']:.4f}  "\n                      f"LPIPS {summary.get(\'lpips\', float(\'nan\')):.4f}  FID {summary.get(\'fid_patch256\', float(\'nan\')):.2f}  "\n                      f"({time.time() - t0:.0f}s)")\n        if method == "average_pool":\n            report.update(reported_baselines(ctx, models))\n        folder = ctx.compare / method\n        rows = [{"set": s, "model": lab, **v} for (s, lab), v in report.items()]\n        write_csv(folder / "quality.csv", rows)\n        write_json(folder / "report.json", rows)\n        flat = [{"set": s, "model": lab, **r} for (s, lab), rr in per_image.items() for r in rr]\n        write_csv(folder / "per_image.csv", flat)\n        write_json(folder / "per_image.json", flat)\n        if method == "average_pool":\n            check_old_checkpoints(ctx, report)\n        quality_by_config(ctx, method, models, val_images, full[method])\n\n\ndef reported_baselines(ctx, models):\n    """Baseline không có checkpoint: dùng số đã báo cáo, nhãn [số cũ]."""\n    label_of = {"SPANV2": "SPANV2", "DISP": "DISP", "PLKSR-Tiny": "PLKSR-Tiny", "SPANV2 + NuSE": "SPANV2 + NuSE k=8 (cũ)",\n                "DISP + NuSE": "DISP + NuSE k=8 (cũ)", "PLKSR-Tiny + NuSE": "PLKSR-Tiny + NuSE k=8 (cũ)"}\n    out, rows = {}, []\n    for key, r in ctx.c.OLD_REPORTED.items():\n        label = label_of[key]\n        if label in models or ("(cũ)" in label and not ctx.c.SHOW_OLD_PLUGIN):\n            continue\n        name = label.replace(" (cũ)", "") + " [số cũ]"\n        out[("DIV2K-valid", name)] = {k: r[k] for k in ("rgb_psnr", "rgb_ssim", "lpips", "lr_max_abs")}\n        rows.append({"model": name, **r})\n        print(f"{name:30s} PSNR {r[\'rgb_psnr\']:.4f}  SSIM {r[\'rgb_ssim\']:.4f} (số đã báo cáo; latency cũ trên {r[\'gpu\']})")\n    write_json(ctx.compare / "reported_rows.json", rows)\n    return out\n\n\ndef check_old_checkpoints(ctx, report):\n    """PSNR đo lại của checkpoint cũ phải khớp summary.json cũ (dưới 0.01 dB)."""\n    keys = {"SPANV2": "SPANV2", "DISP": "DISP", "PLKSR-Tiny": "PLKSR-Tiny", "SPANV2 + NuSE k=8 (cũ)": "SPANV2 + NuSE",\n            "DISP + NuSE k=8 (cũ)": "DISP + NuSE", "PLKSR-Tiny + NuSE k=8 (cũ)": "PLKSR-Tiny + NuSE"}\n    for label, path in ctx.old().items():\n        summary_path = Path(path).parent.parent / "comparison" / "summary.json"\n        mine = report.get(("DIV2K-valid", label))\n        if not summary_path.exists() or mine is None:\n            continue\n        old = json.loads(summary_path.read_text())["reports"].get(keys[label])\n        if old:\n            delta = mine["rgb_psnr"] - old["rgb_psnr"]\n            print(f"{\'OK \' if abs(delta) < 0.01 else \'LỆCH\'} {label:28s} cũ {old[\'rgb_psnr\']:.4f} | "\n                  f"đo lại {mine[\'rgb_psnr\']:.4f} | {delta:+.4f} dB")\n\n\ndef quality_by_config(ctx, method, models, val_images, full):\n    """PSNR theo độ chính xác số (fp32/fp16) và kiểu output (float/uint8), DIV2K valid."""\n    path = ctx.compare / "quality_by_config.json"\n    rows = [r for r in (json.loads(path.read_text()) if path.exists() else []) if r["lr"] != method]\n    for label, (model, k) in models.items():\n        if "(cũ)" in label or name_of(label) not in full:\n            continue\n        for amp in ([False, True] if ctx.device.type == "cuda" else [False]):\n            s, _ = evaluate(model, val_images, method, ctx.device, k=k, amp=amp)\n            p = "fp16" if amp else "fp32"\n            rows += [{"lr": method, "model": label, "precision": p, "output": "float", "psnr": s["rgb_psnr"], "ssim": s["rgb_ssim"]},\n                     {"lr": method, "model": label, "precision": p, "output": "uint8", "psnr": s["rgb_psnr_uint8"], "ssim": float("nan")}]\n    write_csv(ctx.compare / "quality_by_config.csv", rows)\n    write_json(path, rows)\n\n\ndef step_cost(ctx):\n    """Tham số học được, GFLOPs @LR 256², latency GPU (batch 1, CUDA events, xen kẽ nhiều vòng) và CPU."""\n    c = ctx.c\n    torch.backends.cudnn.benchmark = True\n    cost, latency = [], []\n    for method in c.METHODS:\n        models, full = ctx.models_eval(method), ctx.full(method)\n        for label, (model, k) in models.items():\n            cost.append({"lr": method, "model": label, "params": count_params(model, k),\n                         "gflops": gflops(model, c.FLOPS_SIZE, ctx.device, k)})\n        if ctx.device.type == "cuda":\n            for size in c.LATENCY_SIZES:\n                x = torch.rand(1, 3, *size, device=ctx.device)\n                fns = {}\n                for label, (model, k) in models.items():\n                    if "(cũ)" in label or (k is not None and k != model.budgets[0]):\n                        continue\n                    with torch.no_grad():\n                        entries = stage_functions(name_of(label), model, x)\n                    if k is not None and name_of(label) not in full:      # biến thể không chọn: chỉ K lớn nhất\n                        entries = {key: fn for key, fn in entries.items() if key[0].endswith(f" K={model.budgets[-1]}")}\n                    fns.update(entries)\n                for precision in c.LATENCY_PRECISIONS:\n                    times, memory = time_cuda(fns, precision, c.WARMUP, c.ITERS, c.ROUNDS)\n                    for (label, stage), values in times.items():\n                        latency.append({"lr": method, "device": torch.cuda.get_device_name(0), "lr_size": f"{size[0]}x{size[1]}",\n                                        "precision": precision, "model": label, "stage": stage,\n                                        "median_ms": statistics.median(values), "p90_ms": sorted(values)[int(0.9 * len(values))],\n                                        "peak_mem_mb": memory[(label, stage)]})\n        if c.CPU_LATENCY_SIZE:\n            x = torch.rand(1, 3, *c.CPU_LATENCY_SIZE)\n            for label, (model, k) in models.items():\n                if "(cũ)" in label or name_of(label) not in full:\n                    continue\n                cpu_model = copy.deepcopy(model).cpu()\n                with torch.no_grad():\n                    values = time_cpu(lambda: cpu_model(x, k) if k else cpu_model(x), c.CPU_ITERS)\n                latency.append({"lr": method, "device": f"cpu ({torch.get_num_threads()} threads)",\n                                "lr_size": f"{c.CPU_LATENCY_SIZE[0]}x{c.CPU_LATENCY_SIZE[1]}", "precision": "fp32",\n                                "model": label, "stage": "total", "median_ms": statistics.median(values),\n                                "p90_ms": sorted(values)[int(0.9 * len(values))], "peak_mem_mb": float("nan")})\n    for r in ctx.load_list("reported_rows.json"):\n        cost.append({"lr": "average_pool", "model": r["model"], "params": r["params"], "gflops": r["gflops"]})\n    write_csv(ctx.compare / "cost.csv", cost)\n    write_json(ctx.compare / "cost.json", cost)\n    if latency:\n        write_csv(ctx.compare / "latency.csv", latency)\n    write_json(ctx.compare / "latency.json", latency)\n    print(len(cost), "dòng chi phí,", len(latency), "dòng latency"\n          + ("" if ctx.device.type == "cuda" else " (không có GPU: chỉ đo CPU)"))\n',
    'nuse_e/pipeline/figures.py': '"""Bước curves (đường học) và figures (hình so sánh)."""\nimport json\nimport statistics\n\nimport torch\n\nfrom ..data.images import ValImages, as_float\nfrom ..evaluation.metrics import psnr\nfrom ..operators import degrade\nfrom .context import name_of\nfrom .report import Tables\n\ndef plt_module():\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    return plt\n\n\ndef step_curves(ctx):\n    """Hình 1: PSNR DIV2K valid (đủ 100 ảnh) theo epoch.\n\n    Nét liền: bản v2 ở mọi exit K (một model). Nét chấm: mỗi biến thể ở exit lớn nhất (mỗi đường một model).\n    """\n    plt, c = plt_module(), ctx.c\n    fig, axes = plt.subplots(1, len(c.METHODS), figsize=(7 * len(c.METHODS), 4.2), squeeze=False)\n    for ax, method in zip(axes[0], c.METHODS):\n        for model in c.MODELS:\n            path = ctx.run_dir(method, model) / "history.json"\n            if not path.exists():\n                continue\n            hist = [r for r in json.loads(path.read_text()) if "validation" in r]\n            if model == "NuSE-E":\n                for k in ctx.arch(method, model)["budgets"]:\n                    ax.plot([r["epoch"] for r in hist], [r["validation"][f"rgb_psnr_K{k}"] for r in hist],\n                            label=f"v2 K={k}", lw=1.6)\n            elif model.startswith("NuSE-E"):\n                k = ctx.arch(method, model)["budgets"][-1]\n                ax.plot([r["epoch"] for r in hist], [r["validation"][f"rgb_psnr_K{k}"] for r in hist],\n                        ":", label=f"{model.split(\'/\')[1]} K={k}", lw=1.4)\n            else:\n                ax.plot([r["epoch"] for r in hist], [r["validation"]["rgb_psnr"] for r in hist], "--", label=model, lw=1.4)\n        ax.set(title=f"LR {method}", xlabel="epoch", ylabel="PSNR DIV2K valid (dB)")\n        ax.grid(alpha=0.3)\n        ax.legend(fontsize=8, title="liền: v2 mọi K | chấm: biến thể, K lớn nhất", title_fontsize=7)\n    plt.tight_layout()\n    fig.savefig(ctx.fig / "1_training_curves.png", dpi=150)\n    plt.close(fig)\n    print("->", ctx.fig / "1_training_curves.png")\n\n\ndef step_figures(ctx):\n    """Hình 2: PSNR theo chi phí; 3: hai loại LR; 4: phân rã latency; 5: từng ảnh; 6: crop trực quan."""\n    import numpy as np\n    from torch.nn import functional as F\n    plt, c, t = plt_module(), ctx.c, Tables(ctx)\n    colors = {"SPANV2": "#2a78d6", "DISP": "#eb6834", "PLKSR-Tiny": "#1baf7a", "Bicubic": "#8a8984"}\n    palette = ["#e34948", "#b3267a", "#8f4fc9", "#d98a1a", "#5c6b73", "#2d9a8f", "#a0522d", "#7a7a1a", "#1f4e8c", "#c71585"]\n    for method in c.METHODS:\n        sel = t.selected.get(method)\n        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))\n        for ax, xkey, xlabel in [(axes[0], "lat", f"latency fp16 @LR {t.ref} (ms)"),\n                                 (axes[1], "gflops", f"GFLOPs @LR {c.FLOPS_SIZE[0]}²")]:\n            nuse = {}\n            for (s, lab), r in t.report[method].items():\n                if s != "DIV2K-valid" or lab == "Bicubic":\n                    continue\n                x = t.lat(method, lab, t.ref, "fp16") if xkey == "lat" else (t.cost(method, lab) or {}).get("gflops")\n                if x is None:\n                    continue\n                if lab.startswith("NuSE-E"):\n                    nuse.setdefault(name_of(lab), []).append((x, r["rgb_psnr"], lab.split("=")[1]))\n                else:\n                    base = lab.split(" +")[0]\n                    marker = "s" if "[số cũ]" in lab else "^" if "(cũ)" in lab or "NuSE k=8" in lab else "o"\n                    ax.scatter(x, r["rgb_psnr"], s=60, color=colors.get(base.replace(" [số cũ]", ""), "#6250d6"), marker=marker)\n                    ax.annotate(lab, (x, r["rgb_psnr"]), textcoords="offset points", xytext=(5, 4), fontsize=8)\n            for i, (name, pts) in enumerate(nuse.items()):\n                pts.sort()\n                main = name in ("NuSE-E", sel)\n                color = palette[i % len(palette)]\n                ax.plot([p[0] for p in pts], [p[1] for p in pts], "-o" if main else ":o", color=color,\n                        lw=1.8 if main else 1.0, ms=6 if main else 4, label=name + (" ★" if name == sel else ""))\n                if main:\n                    for x, y, k in pts:\n                        ax.annotate(f"K={k}", (x, y), textcoords="offset points", xytext=(4, -10), fontsize=8, color=color)\n            ax.set(xlabel=xlabel, ylabel="PSNR DIV2K valid (dB)", title=f"LR {method}")\n            if xkey == "gflops":\n                ax.set_xscale("log")\n            ax.grid(alpha=0.3)\n            if ax.get_legend_handles_labels()[0]:\n                ax.legend(fontsize=8)\n        plt.tight_layout()\n        fig.savefig(ctx.fig / f"2_psnr_vs_cost_{method}.png", dpi=150)\n        plt.close(fig)\n\n    if any(not r["device"].startswith("cpu") for r in t.latency):\n        fig, axes = plt.subplots(1, len(c.METHODS), figsize=(7 * len(c.METHODS), 4.5), squeeze=False)\n        for ax, method in zip(axes[0], c.METHODS):\n            full = ctx.full(method)\n            labels = [lab for (s, lab) in t.report[method] if s == "DIV2K-valid" and lab != "Bicubic"\n                      and "(cũ)" not in lab and "[số cũ]" not in lab and name_of(lab) in full]\n            pos = np.arange(len(labels))\n            pred = [t.lat(method, lab, t.ref, "fp16", "predictor") or t.lat(method, lab, t.ref, "fp16") or 0 for lab in labels]\n            dec = [t.lat(method, lab, t.ref, "fp16", "decoder + consistency") or 0 for lab in labels]\n            ax.bar(pos, pred, 0.6, label="mạng / predictor")\n            ax.bar(pos, dec, 0.6, bottom=pred, label="decoder + consistency (NuSE-E)")\n            ax.set_xticks(pos, labels, rotation=30, ha="right", fontsize=8)\n            ax.set(ylabel="ms (median, fp16)", title=f"Latency @LR {t.ref}, LR {method}")\n            ax.grid(axis="y", alpha=0.3)\n            ax.legend(fontsize=8)\n        plt.tight_layout()\n        fig.savefig(ctx.fig / "4_latency_breakdown.png", dpi=150)\n        plt.close(fig)\n\n    cross = t.cross()\n    labels = [k for k in cross if "(cũ)" not in k]\n    fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))\n    for ax, (key, title) in zip(axes, [("rgb_psnr", "PSNR ↑"), ("rgb_ssim", "SSIM ↑"), ("lpips", "LPIPS ↓"), ("fid_patch256", "FID ↓")]):\n        width = 0.8 / len(c.METHODS)\n        for i, m in enumerate(c.METHODS):\n            vals = [cross[k][m][1].get(key, np.nan) if m in cross[k] else np.nan for k in labels]\n            ax.bar(np.arange(len(labels)) + i * width, vals, width, label=f"LR {m}")\n        ax.set_xticks(np.arange(len(labels)) + width * (len(c.METHODS) - 1) / 2, labels, rotation=35, ha="right", fontsize=8)\n        ax.set_title(title)\n        ax.grid(axis="y", alpha=0.3)\n        if key in ("rgb_psnr", "rgb_ssim"):\n            finite = [cross[k][m][1].get(key) for m in c.METHODS for k in labels if m in cross[k] and cross[k][m][1].get(key) is not None]\n            if finite:\n                ax.set_ylim(min(finite) - (0.5 if key == "rgb_psnr" else 0.02), max(finite) + (0.3 if key == "rgb_psnr" else 0.01))\n    axes[0].legend(fontsize=8)\n    plt.tight_layout()\n    fig.savefig(ctx.fig / "3_avgpool_vs_bicubic.png", dpi=150)\n    plt.close(fig)\n\n    crop = 192\n    val_files = ctx.data()["val_files"]\n    for method in c.METHODS:\n        sel = t.selected.get(method)\n        per_image = ctx.per_image(method)\n        if not sel:\n            continue\n        ours_label = f"{sel} K={ctx.arch(method, sel)[\'budgets\'][-1]}"\n        ours = per_image.get(("DIV2K-valid", ours_label))\n        others = [lab for (s, lab) in per_image if s == "DIV2K-valid" and not lab.startswith("NuSE-E")]\n        v2_label = f"NuSE-E K={ctx.arch(method, \'NuSE-E\')[\'budgets\'][-1]}"\n        if sel != "NuSE-E" and ("DIV2K-valid", v2_label) in per_image:\n            others.append(v2_label)\n        if not ours or not others:\n            continue\n        fig, axes = plt.subplots(1, len(others), figsize=(5.5 * len(others), 4), squeeze=False)\n        for ax, base in zip(axes[0], others):\n            theirs = {r["name"]: r for r in per_image[("DIV2K-valid", base)]}\n            gain = [r["rgb_psnr"] - theirs[r["name"]]["rgb_psnr"] for r in ours]\n            ax.hist(gain, bins=25, color="#2a78d6")\n            ax.axvline(0, color="k", lw=1)\n            ax.set(title=f"{ours_label} − {base}: thắng {sum(g > 0 for g in gain)}/{len(gain)}, "\n                         f"trung vị {statistics.median(gain):+.3f} dB", xlabel="ΔPSNR (dB)")\n            ax.title.set_fontsize(9)\n        plt.tight_layout()\n        fig.savefig(ctx.fig / f"5_per_image_{method}.png", dpi=150)\n        plt.close(fig)\n\n        models = ctx.models_eval(method)\n        base = others[0]\n        theirs = {r["name"]: r for r in per_image[("DIV2K-valid", base)]}\n        ranked = sorted(ours, key=lambda r: r["rgb_psnr"] - theirs[r["name"]]["rgb_psnr"])\n        stems = [p.stem for p in val_files]\n        for pick in (ranked[-1]["name"], ranked[0]["name"]):\n            hr = as_float(ValImages([val_files[stems.index(pick)]])[0][0], ctx.device)\n            lr = degrade(hr, method)\n            with torch.no_grad():\n                outs = {"Bicubic": F.interpolate(lr, scale_factor=4, mode="bicubic", align_corners=False)}\n                for label, (model, k) in models.items():\n                    if label not in (ours_label, *others):\n                        continue\n                    outs[label] = model(lr, k) if k else model(lr)\n            diff = F.avg_pool2d((outs[ours_label] - outs[base]).abs().mean(1, keepdim=True), min(crop, hr.shape[-2]), stride=crop // 4)\n            row, col = divmod(int(diff.flatten().argmax()), diff.shape[-1])\n            top, left = row * (crop // 4), col * (crop // 4)\n            fig, axes = plt.subplots(1, len(outs) + 1, figsize=(3.6 * (len(outs) + 1), 4))\n            for ax, (title, img) in zip(axes, [*outs.items(), ("HR", hr)]):\n                ax.imshow(img[0, :, top:top + crop, left:left + crop].clamp(0, 1).permute(1, 2, 0).cpu().numpy())\n                ax.axis("off")\n                ax.set_title(title if title == "HR" else f"{title}\\n{psnr(img, hr).item():.2f} dB", fontsize=9)\n            fig.suptitle(f"LR {method} | {pick}")\n            plt.tight_layout()\n            fig.savefig(ctx.fig / f"6_visual_{method}_{pick}.png", dpi=150)\n            plt.close(fig)\n    print("Hình ->", ctx.fig)\n',
    'nuse_e/pipeline/prepare.py': '"""Bước setup (dữ liệu, cache, checkpoint cũ) và stats (thống kê PCA, suy kiến trúc)."""\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport torch\n\nfrom ..core.io import write_json\nfrom ..nullspace.architecture import architecture_from_stats\nfrom ..nullspace.statistics import fit_statistics\n\ndef step_setup(ctx):\n    """Tìm dữ liệu, tạo cache, kiểm checkpoint cũ; chép stats/ và runs/ của phiên trước nếu có --resume-from."""\n    c = ctx.c\n    if ctx.args.resume_from:\n        src = Path(ctx.args.resume_from).expanduser().resolve()\n        for sub in ("stats", "runs"):\n            if (src / sub).is_dir():\n                shutil.copytree(src / sub, ctx.out / sub, dirs_exist_ok=True)\n        print("Đã chép stats/ và runs/ từ", src, "(file đã có không bị xoá; run có DONE sẽ không train lại)")\n    ctx.data()\n    protocol = {"epochs": c.EPOCHS, "batch_size": c.BATCH, "repeat": c.REPEAT, "lr": c.LR, "min_lr": c.MIN_LR,\n                "grad_clip": c.GRAD_CLIP}\n    for label, path in ctx.old().items():\n        cfg = torch.load(path, map_location="cpu", weights_only=False)["config"]\n        method = ((cfg.get("train_dataset_options") or {}).get("resize") or {}).get("method", "average_pool")\n        diff = {k: (cfg.get(k), v) for k, v in protocol.items() if cfg.get(k) != v}\n        print(f"{label:28s} {path}\\n   LR = {method}; khác giao thức: {diff or \'không\'}")\n    if not ctx.old():\n        print("Không có checkpoint cũ (--old-runs): baseline dùng số đã báo cáo [số cũ], trần latency dùng số cũ.")\n    print("Thiết bị:", ctx.device, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])\n    print("Output:", ctx.out)\n\n\ndef step_stats(ctx):\n    """PCA null space làm đối xứng D4, Wiener, R² k-fold; rồi suy kiến trúc của mọi biến thể (không chọn tay)."""\n    c = ctx.c\n    for method in c.METHODS:\n        folder = ctx.stats_dir / method\n        stats_path = folder / "stats.pt"\n        if not stats_path.exists():\n            t0 = time.time()\n            stats = fit_statistics(ctx.data()["train_cache"], method, ctx.device, repeat=c.STATS_REPEAT,\n                                     folds=c.STATS_FOLDS, workers=c.WORKERS, seed=c.SEED)\n            folder.mkdir(parents=True, exist_ok=True)\n            torch.save(stats, stats_path)\n            print(f"[{method}] thống kê xong sau {time.time() - t0:.0f}s")\n        stats = torch.load(stats_path, map_location="cpu", weights_only=False)\n        lam = stats["eigenvalues"].mean(0)\n        print(f"\\n=== {method}: d = {stats[\'d\']}, sai lệch bất biến D4 = {stats.get(\'d4_invariance\', float(\'nan\')):.1e}")\n        print("tỉ số λ_i/λ_(i+1):", [round(x, 3) for x in (lam[:-1] / lam[1:]).tolist()])\n        print("R² tuyến tính (trung bình kênh):", [round(x, 3) for x in stats["r2"].mean(0).tolist()])\n        for name in c.NUSE:\n            arch = architecture_from_stats(stats, **ctx.design_of(name))\n            write_json(ctx.arch_path(method, name), arch)\n            print(f"{name:14s} W={arch[\'width\']} {arch[\'block\']:5s} N={sum(arch[\'blocks\'])} alpha={arch[\'alpha\']:7s} | "\n                  f"K = {arch[\'budgets\']} | block = {arch[\'blocks\']} | L_g = {arch[\'linear\']} | "\n                  f"năng lượng = {[round(e, 3) for e in arch[\'retained_energy\']]}")\n',
    'nuse_e/pipeline/report.py': '"""Bước report: bảng tổng hợp, ràng buộc cứng sau train, report.md."""\nfrom ..core.io import write_csv, write_json\nfrom .context import fmt, md_table, name_of\n\nclass Tables:\n    """Truy vấn report / cost / latency đã ghi trên đĩa."""\n\n    def __init__(self, ctx):\n        self.ctx, c = ctx, ctx.c\n        self.report = {m: ctx.report(m) for m in c.METHODS}\n        self.cost_rows, self.latency = ctx.load_list("cost.json"), ctx.load_list("latency.json")\n        self.selected = ctx.selected()\n        self.ref = f"{c.LATENCY_SIZES[-1][0]}x{c.LATENCY_SIZES[-1][1]}"\n\n    def lat(self, method, label, size, precision, stage="total"):\n        for r in self.latency:\n            if (r["lr"], r["model"], r["lr_size"], r["precision"], r["stage"]) == (method, label, size, precision, stage) \\\n                    and not r["device"].startswith("cpu"):\n                return r["median_ms"]\n        return None\n\n    def cost(self, method, label):\n        return next((r for r in self.cost_rows if r["lr"] == method and r["model"] == label), None)\n\n    def cross(self):\n        """NuSE-E (bản v2 và bản được chọn) ở K nhỏ nhất và K lớn nhất, cùng baseline, cho cả hai loại LR."""\n        out = {}\n        for m in self.ctx.c.METHODS:\n            for (s, lab), r in self.report[m].items():\n                key = lab\n                if lab.startswith("NuSE-E"):\n                    name = name_of(lab)\n                    if name not in ("NuSE-E", self.selected.get(m)):\n                        continue\n                    k, budgets = int(lab.split("=")[1]), self.ctx.arch(m, name)["budgets"]\n                    key = f"{name} (K lớn nhất)" if k == budgets[-1] else f"{name} (K nhỏ nhất)" if k == budgets[0] else None\n                if s == "DIV2K-valid" and key:\n                    out.setdefault(key, {})[m] = (lab, r)\n        return out\n\n\ndef step_report(ctx):\n    """Bảng tổng hợp, ràng buộc cứng sau train, so hai loại LR, phân rã latency -> comparison/report.md."""\n    c, t = ctx.c, Tables(ctx)\n    cap_ms = ctx.screen()["cap_ms"]\n    flops = f"{c.FLOPS_SIZE[0]}x{c.FLOPS_SIZE[1]}"\n    parts = []\n    for method in c.METHODS:\n        rows = []\n        for (set_name, label), r in t.report[method].items():\n            if set_name != "DIV2K-valid":\n                continue\n            co = t.cost(method, label)\n            rows.append([label, fmt(r["rgb_psnr"]), fmt(r["rgb_ssim"]), fmt(r.get("y_psnr_shave4")), fmt(r.get("y_ssim_shave4")),\n                         fmt(r.get("lpips")), fmt(r.get("fid_patch256"), 2), f"{r[\'lr_max_abs\']:.1e}",\n                         f"{co[\'params\'] / 1e3:.1f}K" if co else "—", fmt(co["gflops"], 2) if co else "—",\n                         fmt(t.lat(method, label, t.ref, "fp32"), 2), fmt(t.lat(method, label, t.ref, "fp16"), 2)])\n        parts += [f"## LR = {method}, DIV2K valid", md_table(\n            ["model", "PSNR ↑", "SSIM ↑", "Y-PSNR ↑", "Y-SSIM ↑", "LPIPS ↓", "FID ↓", "max|A(SR)−y|", "params",\n             f"GFLOPs@{c.FLOPS_SIZE[0]}²", f"ms fp32 @{t.ref}", f"ms fp16 @{t.ref}"], rows)]\n        bench = [(s, lab, r) for (s, lab), r in t.report[method].items() if s != "DIV2K-valid"]\n        if bench:\n            sets = sorted({s for s, _, _ in bench})\n            labels = list(dict.fromkeys(lab for _, lab, _ in bench))\n            parts += [f"### Benchmark (Y-PSNR / Y-SSIM, shave 4), LR = {method}", md_table(\n                ["model", *sets], [[lab, *[next((f"{r[\'y_psnr_shave4\']:.2f} / {r[\'y_ssim_shave4\']:.4f}"\n                                                 for s2, l2, r in bench if s2 == s and l2 == lab), "—") for s in sets]]\n                                   for lab in labels])]\n        write_json(ctx.compare / method / "summary.json",\n                   {"reports": {f"{s} | {lab}": r for (s, lab), r in t.report[method].items()},\n                    "cost": [r for r in t.cost_rows if r["lr"] == method],\n                    "architecture": {n: ctx.arch(method, n) for n in c.NUSE if ctx.arch_path(method, n).exists()},\n                    "selected": t.selected.get(method)})\n\n    constraint_rows = []\n    for method in c.METHODS:\n        for name in c.NUSE:\n            if not any(s == "DIV2K-valid" and name_of(lab) == name for (s, lab) in t.report[method]):\n                continue\n            arch = ctx.arch(method, name)\n            ks = arch["budgets"]\n            psnr = [t.report[method][("DIV2K-valid", f"{name} K={k}")]["rgb_psnr"] for k in ks]\n            params = [(t.cost(method, f"{name} K={k}") or {}).get("params") for k in ks]\n            ms = [t.lat(method, f"{name} K={k}", flops, "fp32") for k in ks]\n            ms_known = [v for v in ms if v is not None]\n            consistency = max(t.report[method][("DIV2K-valid", f"{name} K={k}")]["lr_max_abs"] for k in ks)\n            checks = {"params ≤ trần": params[-1] is not None and params[-1] <= c.PARAM_CAP,\n                      "latency ≤ trần": ms[-1] is None or ms[-1] <= cap_ms,\n                      "params tăng theo K": all(b > a for a, b in zip(params, params[1:]) if a and b),\n                      "latency tăng theo K": all(b > a for a, b in zip(ms_known, ms_known[1:])),\n                      "PSNR tăng theo K": all(b > a for a, b in zip(psnr, psnr[1:])),\n                      "A(SR)=y": consistency < c.CONSISTENCY_TOL}\n            constraint_rows.append({"lr": method, "model": name, "selected": name == t.selected.get(method),\n                                    "W": arch["width"], "block": arch["block"], "N": sum(arch["blocks"]), "alpha": arch["alpha"],\n                                    "K": str(ks), "params_kmax": params[-1], "ms_kmax_fp32_256": ms[-1],\n                                    **{f"psnr_K{k}": p for k, p in zip(ks, psnr)}, "psnr_kmax": psnr[-1],\n                                    "lr_max_abs": consistency, **{f"ok: {k}": v for k, v in checks.items()},\n                                    "all_ok": all(checks.values())})\n    write_csv(ctx.compare / "constraints.csv", constraint_rows)\n    parts += ["## Các biến thể NuSE-E và ràng buộc cứng (sau khi train)", md_table(\n        ["LR", "model", "W", "block", "N", "alpha", "K", "params K max", "ms fp32 @256² K max", "PSNR theo K",\n         "max|A(SR)−y|", "ràng buộc"],\n        [[r["lr"], ("★ " if r["selected"] else "") + r["model"], r["W"], r["block"], r["N"], r["alpha"], r["K"],\n          f"{r[\'params_kmax\']:,}" if r["params_kmax"] else "—", fmt(r["ms_kmax_fp32_256"], 2),\n          " / ".join(f"{r[k]:.3f}" for k in r if k.startswith("psnr_K")), f"{r[\'lr_max_abs\']:.1e}",\n          "đạt hết" if r["all_ok"] else "VI PHẠM: " + ", ".join(k[4:] for k in r if k.startswith("ok: ") and not r[k])]\n         for r in constraint_rows])]\n    warnings = [f"CẢNH BÁO: bản được chọn {r[\'model\']} / {r[\'lr\']} vi phạm ràng buộc, xem cột cuối."\n                for r in constraint_rows if r["selected"] and not r["all_ok"]]\n\n    rows = []\n    for key, per in t.cross().items():\n        row = [key]\n        for m in c.METHODS:\n            lab, r = per.get(m, (None, None))\n            row.append("—" if r is None else f"{r[\'rgb_psnr\']:.3f} / {r[\'rgb_ssim\']:.4f} / {fmt(r.get(\'lpips\'))} / "\n                       f"{fmt(r.get(\'fid_patch256\'), 1)}" + (f" ({lab.split(\' \')[1]})" if lab.startswith("NuSE-E") else ""))\n        rows.append(row)\n    parts += ["## So sánh hai loại LR (DIV2K valid)", md_table(["model", *[f"{m}: PSNR / SSIM / LPIPS / FID" for m in c.METHODS]], rows)]\n\n    if t.latency:\n        rows = []\n        for m in c.METHODS:\n            for name in dict.fromkeys(n for n in ("NuSE-E", t.selected.get(m)) if n):\n                for k in ctx.arch(m, name)["budgets"]:\n                    for size in [f"{h}x{w}" for h, w in c.LATENCY_SIZES]:\n                        for p in c.LATENCY_PRECISIONS:\n                            pr, dc, tt = (t.lat(m, f"{name} K={k}", size, p, s) for s in ("predictor", "decoder + consistency", "total"))\n                            if tt:\n                                rows.append([m, name, k, size, p, fmt(pr, 3), fmt(dc, 3), fmt(tt, 3), f"{dc / tt:.0%}"])\n        if rows:\n            parts += ["## NuSE-E: phân rã latency (median ms, GPU)", md_table(\n                ["LR", "model", "K", "kích thước LR", "precision", "predictor", "decoder + consistency", "total", "tỉ lệ decoder"], rows)]\n    text = "# Kết quả NuSE-E\\n\\n" + "\\n\\n".join(parts + warnings) + "\\n"\n    (ctx.compare / "report.md").write_text(text)\n    print(text)\n    print("->", ctx.compare / "report.md")\n',
    'nuse_e/pipeline/selection.py': '"""Bước select: chọn biến thể tốt nhất mỗi loại LR."""\nimport torch\n\nfrom ..core.io import write_csv, write_json\n\ndef step_select(ctx):\n    """Chọn biến thể có PSNR K lớn nhất cao nhất (bằng nhau thì ít tham số hơn); bản v2 luôn giữ để đối chiếu."""\n    c = ctx.c\n    screen = {r["model"]: r for r in ctx.screen()["rows"]}\n    selected, rows = {}, []\n    for method in c.METHODS:\n        cands = []\n        for name in c.NUSE:\n            if name not in screen or not screen[name]["admissible"] or not ctx.is_done(method, name):\n                continue\n            ck = torch.load(ctx.run_dir(method, name) / "best.pt", map_location="cpu", weights_only=False)\n            rec = next(r for r in ck["history"] if r["epoch"] == ck["epoch"])\n            arch = ck["arch"]\n            # Checkpoint của bản v2 cũ không lưu width/block/alpha: dùng mặc định của NuSEE.\n            row = {"lr": method, "model": name, "width": arch.get("width", 32), "block": arch.get("block", "plain"),\n                   "blocks": sum(arch["blocks"]), "alpha": arch.get("alpha", "uniform"), "budgets": str(arch["budgets"]),\n                   "best_epoch": ck["epoch"], "params_kmax": screen[name]["params_kmax"],\n                   "psnr_mean_K": rec["validation"]["rgb_psnr"],\n                   **{f"psnr_K{k}": rec["validation"][f"rgb_psnr_K{k}"] for k in arch["budgets"]}}\n            rows.append(row)\n            cands.append((rec["validation"][f"rgb_psnr_K{arch[\'budgets\'][-1]}"], -row["params_kmax"], name))\n        if cands:\n            best = max(p for p, _, _ in cands)\n            tied = [cand for cand in cands if cand[0] >= best - c.SELECT_TOL_DB]   # trong ngưỡng nhiễu: coi là hoà\n            selected[method] = max(tied, key=lambda cand: (cand[1], cand[0]))[2]   # ít tham số hơn, rồi PSNR cao hơn\n        else:\n            selected[method] = None\n        print(f"[{method}] chọn: {selected[method]} (chênh dưới {c.SELECT_TOL_DB} dB coi là hoà, lấy bản ít tham số hơn)")\n        for cand in sorted(cands, reverse=True):\n            print(f"   {cand[2]:14s} PSNR K max {cand[0]:.4f}  tham số {-cand[1]:,}")\n    write_csv(ctx.compare / "variants.csv", rows)\n    write_json(ctx.compare / "selected.json", selected)\n',
    'nuse_e/pipeline/training.py': '"""Bước train: chạy mọi run còn thiếu, mỗi GPU một tiến trình; pha A rồi pha B."""\nimport os\nimport subprocess\nimport sys\nimport time\n\nimport torch\n\nfrom ..core.io import write_json\nfrom .context import PACKAGE_PARENT\n\ndef step_train(ctx):\n    """Train mọi run còn thiếu: pha A (mọi biến thể, LR average pool), rồi pha B (bicubic cho biến thể tốt nhất)."""\n    c, data = ctx.c, ctx.data()\n    admissible = ctx.admissible(None)\n    priority = {m: i for i, m in enumerate(c.MODELS)}\n    no_train = set(c.BASELINES)\n\n    def wanted(method, model):\n        if model.startswith("NuSE-E"):\n            if model not in admissible:\n                return False\n            return method == "average_pool" or model == "NuSE-E" or "average_pool" not in c.METHODS\n        return model not in no_train and not ctx.uses_old(method, model)\n\n    def best_psnr_kmax(method, model):\n        ck = torch.load(ctx.run_dir(method, model) / "best.pt", map_location="cpu", weights_only=False)\n        kmax = ck["arch"]["budgets"][-1]\n        rec = next(r for r in ck["history"] if r["epoch"] == ck["epoch"])\n        return rec["validation"][f"rgb_psnr_K{kmax}"]\n\n    def job_config(method, model):\n        nuse = model.startswith("NuSE-E")\n        cfg = dict(model=model, method=method, run_dir=str(ctx.run_dir(method, model)),\n                   train_files=[str(p) for p in data["train_files"]], val_files=[str(p) for p in data["val_files"]],\n                   train_cache=str(data["train_cache"]),\n                   stats=str(ctx.stats_dir / method / "stats.pt") if nuse else None,\n                   arch=str(ctx.arch_path(method, model)) if nuse else None,\n                   epochs=c.EPOCHS, batch=c.BATCH, repeat=c.REPEAT, lr=c.LR, min_lr=c.MIN_LR, grad_clip=c.GRAD_CLIP,\n                   val_every=c.VAL_EVERY, checkpoint_every=c.CHECKPOINT_EVERY, workers=c.WORKERS, seed=c.SEED,\n                   amp=c.AMP, quick_val_images=c.QUICK_VAL_IMAGES, quick_val_crop=c.QUICK_VAL_CROP,\n                   device="cuda" if ctx.device.type == "cuda" else "cpu", max_iters=c.MAX_ITERS,\n                   deadline=ctx.args.deadline)\n        path = ctx.run_dir(method, model) / "config.json"\n        write_json(path, cfg)\n        return path\n\n    jobs = sorted([(m, mdl) for m in c.METHODS for mdl in c.MODELS if wanted(m, mdl) and not ctx.is_done(m, mdl)],\n                  key=lambda j: (priority[j[1]], j[0] != "average_pool"))\n    phase_b = "bicubic" in c.METHODS and "average_pool" in c.METHODS and c.PHASE_B_TOP > 0\n    print("Đã xong từ trước (DONE):", [f"{mdl} / {m}" for m in c.METHODS for mdl in c.MODELS if ctx.is_done(m, mdl)] or "không có")\n    print("Cần train:", jobs or "không còn run nào")\n    if ctx.args.gpus:\n        slots = ctx.args.gpus.split(",")\n    else:\n        slots = [str(i) for i in range(torch.cuda.device_count())] if ctx.device.type == "cuda" else ["cpu"]\n    start, active, shown, cut = time.time(), {}, {}, False\n    while jobs or active or phase_b:\n        if not jobs and not active and phase_b:\n            phase_b = False\n            done_a = [n for n in c.NUSE if n != "NuSE-E" and n in admissible and ctx.is_done("average_pool", n)]\n            ranked = sorted(done_a, key=lambda n: -best_psnr_kmax("average_pool", n))\n            jobs = [("bicubic", n) for n in ranked[:c.PHASE_B_TOP] if not ctx.is_done("bicubic", n)]\n            print("Pha B (LR bicubic) cho biến thể tốt nhất pha A:", jobs or "không có")\n        for slot in [s for s in slots if s not in active]:\n            if not jobs:\n                break\n            if c.MAX_HOURS and (time.time() - start) / 3600 > c.MAX_HOURS:\n                print(f"Quá {c.MAX_HOURS} giờ: không nhận run mới. Chạy lại bước train để tiếp tục.")\n                jobs, phase_b = [], False\n                break\n            if ctx.args.deadline and time.time() > ctx.args.deadline:\n                print("Đã tới --deadline: không nhận run mới. Chạy lại bước train để tiếp tục.")\n                jobs, phase_b, cut = [], False, True\n                break\n            method, model = jobs.pop(0)\n            env = dict(os.environ, PYTHONPATH=os.pathsep.join(filter(None, [\n                str(PACKAGE_PARENT), ctx.paths.get("experiment"), os.environ.get("PYTHONPATH")])))\n            if slot != "cpu":\n                env["CUDA_VISIBLE_DEVICES"] = slot\n            config_path = job_config(method, model)\n            log = open(ctx.run_dir(method, model) / "stdout.log", "a")\n            active[slot] = (method, model, subprocess.Popen(\n                [sys.executable, "-u", "-m", "nuse_e.training.run", str(config_path)],\n                env=env, stdout=log, stderr=subprocess.STDOUT), log)\n            print(f"[{slot}] bắt đầu {model} / {method}")\n        time.sleep(1 if c.SMOKE else 30)\n        for slot, (method, model, proc, log) in list(active.items()):\n            tail = ctx.run_dir(method, model) / "train.log"\n            text = tail.read_text().strip() if tail.exists() else ""\n            last = text.splitlines()[-1] if text else ""\n            if last and shown.get(slot) != last:\n                print(f"[{slot}] {model} / {method}: {last}", flush=True)\n                shown[slot] = last\n            if proc.poll() is not None:\n                log.close()\n                del active[slot]\n                if proc.returncode != 0:\n                    print((ctx.run_dir(method, model) / "stdout.log").read_text()[-3000:])\n                    sys.exit(f"{model} / {method} lỗi (mã {proc.returncode})")\n    # Run tạm dừng vì --deadline (không có DONE) hay chưa kịp bắt đầu: ghi lại để notebook biết phải chạy tiếp.\n    pending = [f"{mdl} / {m}" for m in c.METHODS for mdl in c.MODELS if not ctx.is_done(m, mdl) and (\n        wanted(m, mdl) or (ctx.run_dir(m, mdl) / "last.pt").exists())]\n    if cut and not pending:\n        pending = ["pha B (LR bicubic) chưa bắt đầu"]\n    write_json(ctx.compare / "train_pending.json", pending)\n    if pending:\n        print("Chưa xong (chạy lại bước train để tiếp tục từ last.pt):", pending)\n    print(f"Train {\'xong\' if not pending else \'tạm dừng\'} sau {(time.time() - start) / 3600:.2f} giờ")\n',
    'nuse_e/training/__init__.py': '"""Train một run: trainer.train(cfg, log). Chạy từ dòng lệnh: python -m nuse_e.training.run <config.json>."""\n',
    'nuse_e/training/run.py': '"""Train một run trong một tiến trình riêng (một GPU).\n\nDùng: python -m nuse_e.training.run <run_dir>/config.json\nBước `train` của pipeline gọi lệnh này cho từng run; có thể gọi tay để chạy lại một run.\n"""\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nfrom .trainer import train\n\n\ndef main(config_path):\n    cfg = json.loads(Path(config_path).read_text())\n    log_path = Path(cfg["run_dir"]) / "train.log"\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n\n    def log(message):\n        line = f"{time.strftime(\'%H:%M:%S\')} {message}"\n        print(line, flush=True)\n        with log_path.open("a") as stream:\n            stream.write(line + "\\n")\n\n    log(f"start {cfg[\'model\']} / {cfg[\'method\']} on {cfg[\'device\']}")\n    train(cfg, log)\n    log("done")\n\n\nif __name__ == "__main__":\n    main(sys.argv[1])\n',
    'nuse_e/training/trainer.py': '"""Vòng train một run: AdamW, cosine theo epoch, validate, best.pt / last.pt / DONE."""\nimport json\nimport os\nimport statistics\nimport time\nfrom pathlib import Path\n\nimport torch\n\nfrom ..core.constants import S\nfrom ..core.io import seed_everything, write_json\nfrom ..data.cache import FlatImages\nfrom ..data.images import ValImages\nfrom ..data.loaders import RamCropLoader, train_loader\nfrom ..evaluation.evaluate import evaluate\nfrom ..models.nuse_e import NuSEE\nfrom ..models.registry import budgets_of, build_model\nfrom ..operators import degrade\n\ndef quick_images(val, count, crop):\n    """First `count` validation images, centre-cropped to crop x crop: fast per-epoch progress check."""\n    out = []\n    for i in range(min(count, len(val))):\n        hr, name = val[i]\n        c = min(crop, hr.shape[-2], hr.shape[-1]) // S * S\n        top, left = (hr.shape[-2] - c) // 2 // S * S, (hr.shape[-1] - c) // 2 // S * S\n        out.append((hr[:, top:top + c, left:left + c].contiguous(), name))\n    return out\n\n\ndef progress_text(metrics, model):\n    if isinstance(model, NuSEE):\n        return " | ".join(f"K{k}: {metrics[f\'rgb_psnr_K{k}\']:.3f} dB / {metrics[f\'rgb_ssim_K{k}\']:.4f}" for k in model.budgets)\n    return f"{metrics[\'rgb_psnr\']:.3f} dB / {metrics[\'rgb_ssim\']:.4f}"\n\n\n@torch.no_grad()\ndef validate(model, images, method, device):\n    out = {}\n    for k in budgets_of(model):\n        summary, _ = evaluate(model, images, method, device, k=k)\n        out["rgb_psnr" if k is None else f"rgb_psnr_K{k}"] = summary["rgb_psnr"]\n        out["rgb_ssim" if k is None else f"rgb_ssim_K{k}"] = summary["rgb_ssim"]\n    if isinstance(model, NuSEE):\n        out["rgb_psnr"] = statistics.fmean(out[f"rgb_psnr_K{k}"] for k in model.budgets)\n        out["rgb_ssim"] = statistics.fmean(out[f"rgb_ssim_K{k}"] for k in model.budgets)\n    return out\n\n\ndef train(cfg, log=print):\n    """One run. cfg keys: model, method, run_dir, train_files, val_files, stats, arch, epochs, batch,\n    repeat, lr, min_lr, grad_clip, val_every, checkpoint_every, workers, seed, amp, device, max_iters, deadline."""\n    device = torch.device(cfg["device"])\n    run_dir = Path(cfg["run_dir"])\n    run_dir.mkdir(parents=True, exist_ok=True)\n    seed_everything(cfg["seed"])\n    stats = torch.load(cfg["stats"], map_location="cpu", weights_only=False) if cfg.get("stats") else None\n    arch = json.loads(Path(cfg["arch"]).read_text()) if cfg.get("arch") else None\n    model = build_model(cfg["model"], cfg["method"], stats, arch).to(device)\n    params = [p for p in model.parameters() if p.requires_grad]\n    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=0)\n    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"], eta_min=cfg["min_lr"])\n    scaler = torch.amp.GradScaler("cuda", enabled=cfg["amp"] and device.type == "cuda")\n    if cfg.get("train_cache"):                           # flat uint8 cache in RAM: no image decoding per crop\n        loader = RamCropLoader(FlatImages(cfg["train_cache"]), cfg["batch"], cfg["seed"], repeat=cfg["repeat"])\n    else:\n        data, dl = train_loader([Path(p) for p in cfg["train_files"]], batch=cfg["batch"], workers=cfg["workers"],\n                                seed=cfg["seed"], repeat=cfg["repeat"])\n\n        class _Wrapped:\n            def set_epoch(self, epoch):\n                data.set_epoch(epoch)\n                dl.generator.manual_seed(cfg["seed"] + epoch)\n\n            def __iter__(self):\n                for hr, idx in dl:\n                    yield hr.mul(255).round().byte(), idx\n        loader = _Wrapped()\n    val = ValImages([Path(p) for p in cfg["val_files"]])\n    val_images = [val[i] for i in range(len(val))]\n    quick = quick_images(val, cfg.get("quick_val_images", 10), cfg.get("quick_val_crop", 480))\n    start, best, history = 1, -float("inf"), []\n    last = run_dir / "last.pt"\n    if last.exists():\n        ck = torch.load(last, map_location="cpu", weights_only=False)\n        model.load_state_dict(ck["model"])\n        opt.load_state_dict(ck["optimizer"])\n        sched.load_state_dict(ck["scheduler"])\n        scaler.load_state_dict(ck["scaler"])\n        start, best, history = ck["epoch"] + 1, ck["best"], ck["history"]\n        log(f"resume from epoch {ck[\'epoch\']}")\n    meta = {"model": cfg["model"], "method": cfg["method"], "protocol": {k: cfg[k] for k in (\n        "epochs", "batch", "repeat", "lr", "min_lr", "grad_clip", "seed", "amp")}}\n    t_run, slowest = time.time(), 0.0\n    for epoch in range(start, cfg["epochs"] + 1):\n        model.train()\n        loader.set_epoch(epoch)\n        total, count, t0, t_data = 0.0, 0, time.time(), 0.0\n        t_step = time.time()\n        for step, (hr, _) in enumerate(loader, 1):\n            t_data += time.time() - t_step\n            hr = hr.to(device, non_blocking=True).float().div_(255)\n            with torch.no_grad():\n                lr = degrade(hr, cfg["method"])\n            with torch.autocast(device.type, dtype=torch.float16, enabled=cfg["amp"] and device.type == "cuda"):\n                loss = model.loss(lr, hr)\n            loss = loss.float()\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f"non-finite loss at epoch {epoch} step {step}")\n            opt.zero_grad(set_to_none=True)\n            scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(params, cfg["grad_clip"])\n            scaler.step(opt)\n            scaler.update()\n            total += loss.item() * hr.shape[0]\n            count += hr.shape[0]\n            t_step = time.time()\n            if cfg.get("max_iters") and step >= cfg["max_iters"]:\n                break\n        record = {"epoch": epoch, "lr": opt.param_groups[0]["lr"], "train_loss": total / count,\n                  "epoch_seconds": time.time() - t0, "data_seconds": t_data}\n        sched.step()\n        # Progress check every epoch on a few centre crops: no gradient, no RNG use, so training is unchanged.\n        record["quick"] = validate(model, quick, cfg["method"], device)\n        model.train()\n        improved = False\n        if epoch % cfg["val_every"] == 0 or epoch == cfg["epochs"]:\n            record["validation"] = validate(model, val_images, cfg["method"], device)\n            improved = record["validation"]["rgb_psnr"] > best\n            best = max(best, record["validation"]["rgb_psnr"])\n        history.append(record)\n        state = {"nuse_e": meta, "epoch": epoch, "model": model.state_dict(), "optimizer": opt.state_dict(),\n                 "scheduler": sched.state_dict(), "scaler": scaler.state_dict(), "best": best,\n                 "history": history, "config": cfg, "stats": stats, "arch": arch}\n        tmp = run_dir / "last.pt.tmp"\n        torch.save(state, tmp)\n        os.replace(tmp, last)\n        if improved:\n            torch.save(state, run_dir / "best.pt")\n        if epoch % cfg["checkpoint_every"] == 0:\n            torch.save(state, run_dir / f"epoch_{epoch:04d}.pt")\n        write_json(run_dir / "history.json", history)\n        msg = (f"epoch {epoch}/{cfg[\'epochs\']} loss={record[\'train_loss\']:.6f} "\n               f"({record[\'epoch_seconds\']:.0f}s, chờ dữ liệu {t_data:.0f}s) "\n               f"| nhanh ({len(quick)} crop): {progress_text(record[\'quick\'], model)}")\n        if "validation" in record:\n            msg += f" || DIV2K valid đủ {len(val_images)} ảnh: {progress_text(record[\'validation\'], model)}"\n        log(msg)\n        # Sắp hết giờ (--deadline, vd. giới hạn 12 giờ của Kaggle): dừng sau epoch vừa lưu last.pt, không ghi DONE.\n        # Chạy lại bước train (cùng output) thì train tiếp từ last.pt, kết quả như chạy một mạch.\n        slowest = max(slowest, time.time() - t0)\n        if cfg.get("deadline") and epoch < cfg["epochs"] and time.time() + 1.5 * slowest > cfg["deadline"]:\n            log(f"tạm dừng sau epoch {epoch}/{cfg[\'epochs\']} vì sắp hết giờ; chạy lại bước train để tiếp tục")\n            return\n    (run_dir / "DONE").write_text(f"{time.time() - t_run:.0f}s\\n")\n',
    'experiments/c1_rep_branches/experiment.py': '"""Cách 1: conv ba nhánh cộng lại khi train, gộp thành một Conv3x3 khi suy luận; so với e80 cùng chi phí suy luận.\n\nPipeline nuse_e đọc các tên viết hoa ở đây thay cho giá trị trong nuse_e/config.py\n(python -m nuse_e.pipeline ... --experiment experiments/c1_rep_branches).\n"""\nfrom nuse_e.config import VARIANTS as BASE\n\nIMPL = "rep_branches.model:RepBranchNuSEE"\nE80 = BASE["e80"]                                       # 7 block (5, 1, 1), exit K = 5 / 9 / 15\n\nVARIANTS = {\n    "e80": E80,                                         # tham chiếu; có DONE sẵn nếu --resume-from output v3\n    "e80rep": dict(E80, block="rep"),                   # 3x3 + 1x1 + identity (RepVGG; RepConv có sẵn của nuse_e)\n    "e80acb": dict(E80, block="acb", impl=IMPL),        # 3x3 + 1x3 + 3x1 (ACNet)\n    "e80ecb": dict(E80, block="ecb", impl=IMPL),        # 3x3 + (1x1 -> 3x3) + (1x1 -> Sobel/Laplacian) (ECBSR)\n}\nMETHODS = ["average_pool", "bicubic"]                   # pha A: LR average pool, rồi pha B: LR bicubic\nPHASE_B_TOP = len(VARIANTS)                             # pha B cho mọi biến thể (không chỉ N bản tốt nhất pha A)\n',
    'experiments/c1_rep_branches/rep_branches/__init__.py': '"""Cách 1: conv ba nhánh cộng lại khi train, gộp thành một Conv3x3 khi suy luận (re-parameterisation).\n\n    branches  ACBConv, ECBConv: conv ba nhánh và phép gộp\n    model     RepBranchNuSEE: NuSE-E e80 với mỗi Conv3x3 của trunk là conv ba nhánh\n"""\nfrom .branches import ACBConv, ECBConv\nfrom .model import BranchResBlock, RepBranchNuSEE\n\n__all__ = ["ACBConv", "ECBConv", "BranchResBlock", "RepBranchNuSEE"]\n',
    'experiments/c1_rep_branches/rep_branches/branches.py': '"""Conv3x3 ba nhánh khi train; `fuse()` trả về một nn.Conv2d 3x3 cho output giống hệt (sai số làm tròn).\n\n    ACBConv     3x3 + 1x3 + 3x1                          ACNet (Ding et al., ICCV 2019)\n    ECBConv     3x3 + (1x1 -> 3x3) + (1x1 -> lọc cạnh)   ECBSR (Zhang et al., ACM MM 2021), rút còn ba nhánh\n    (bản RepVGG 3x3 + 1x1 + identity đã có sẵn: RepConv trong nuse_e/models/blocks.py)\n\nMọi nhánh đều tuyến tính, không BN, nên tổng của chúng vẫn là một phép conv 3x3. Nhánh có conv 1x1 đứng trước conv 3x3\nđược pad bằng bias của conv 1x1 thay vì 0 (như ECBSR): khi đó viền ảnh cũng khớp đúng với conv đã gộp, vì conv gộp\nnhìn thấy x = 0 ngoài ảnh, và conv 1x1 biến x = 0 thành đúng bias của nó.\n"""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n# Sobel-x, Sobel-y, Laplacian: đạo hàm bậc 1 theo hai hướng và bậc 2 (nhánh cạnh của ECBSR).\nEDGE_MASKS = torch.tensor([[[1., 0., -1.], [2., 0., -2.], [1., 0., -1.]],\n                           [[1., 2., 1.], [0., 0., 0.], [-1., -2., -1.]],\n                           [[0., 1., 0.], [1., -4., 1.], [0., 1., 0.]]])\n\n\ndef conv3x3(weight, bias):\n    conv = nn.Conv2d(weight.shape[1], weight.shape[0], 3, padding=1).to(weight)\n    conv.weight.copy_(weight)\n    conv.bias.copy_(bias)\n    return conv\n\n\ndef pad_with_bias(t, bias):\n    """Pad 1 pixel mỗi cạnh, giá trị pad = bias của từng kênh (tương đương conv 1x1 nhìn thấy x = 0 ngoài ảnh)."""\n    b = bias.view(1, -1, 1, 1)\n    return F.pad(t - b, (1, 1, 1, 1)) + b\n\n\nclass ACBConv(nn.Module):\n    """3x3 + 1x3 + 3x1: hai nhánh dải tăng cường hàng giữa và cột giữa của kernel ("bộ xương" của kernel)."""\n\n    def __init__(self, width):\n        super().__init__()\n        self.square = nn.Conv2d(width, width, 3, padding=1)\n        self.hor = nn.Conv2d(width, width, (1, 3), padding=(0, 1))\n        self.ver = nn.Conv2d(width, width, (3, 1), padding=(1, 0))\n\n    def forward(self, x):\n        return self.square(x) + self.hor(x) + self.ver(x)\n\n    @torch.no_grad()\n    def fuse(self):\n        weight = self.square.weight.clone()\n        weight[:, :, 1:2, :] += self.hor.weight\n        weight[:, :, :, 1:2] += self.ver.weight\n        return conv3x3(weight, self.square.bias + self.hor.bias + self.ver.bias)\n\n\nclass ECBConv(nn.Module):\n    """3x3 + (1x1 nở ra `expand` lần -> 3x3 thu lại) + (1x1 -> depthwise lọc cạnh).\n\n    Nhánh cạnh: mỗi kênh một tổ hợp học được của Sobel-x, Sobel-y, Laplacian (ECBSR có ba nhánh riêng cho ba bộ lọc;\n    ở đây gộp chung một conv 1x1 để giữ đúng ba nhánh). Tỉ lệ khởi tạo nhỏ như ECBSR.\n    """\n\n    def __init__(self, width, expand=2):\n        super().__init__()\n        self.square = nn.Conv2d(width, width, 3, padding=1)\n        self.expand = nn.Conv2d(width, expand * width, 1)\n        self.squeeze = nn.Conv2d(expand * width, width, 3)              # pad bằng tay (pad_with_bias)\n        self.reduce = nn.Conv2d(width, width, 1)\n        self.edge_scale = nn.Parameter(torch.randn(width, len(EDGE_MASKS)) * 1e-3)\n        self.edge_bias = nn.Parameter(torch.zeros(width))\n        self.register_buffer("edge_masks", EDGE_MASKS.clone())\n\n    def edge_kernel(self):\n        """(width, 1, 3, 3): kernel depthwise của nhánh cạnh."""\n        return torch.einsum("cm,mhw->chw", self.edge_scale, self.edge_masks).unsqueeze(1)\n\n    def forward(self, x):\n        out = self.square(x)\n        out = out + self.squeeze(pad_with_bias(self.expand(x), self.expand.bias))\n        edge = pad_with_bias(self.reduce(x), self.reduce.bias)\n        return out + F.conv2d(edge, self.edge_kernel(), self.edge_bias, groups=edge.shape[1])\n\n    @torch.no_grad()\n    def fuse(self):\n        w1, b1 = self.expand.weight[..., 0, 0], self.expand.bias            # (m, c), (m,)\n        w2, b2 = self.squeeze.weight, self.squeeze.bias                     # (o, m, 3, 3), (o,)\n        weight = self.square.weight + torch.einsum("omhw,mc->ochw", w2, w1)\n        bias = self.square.bias + b2 + torch.einsum("omhw,m->o", w2, b1)\n        k, w3, b3 = self.edge_kernel()[:, 0], self.reduce.weight[..., 0, 0], self.reduce.bias   # (o,3,3), (o,c), (o,)\n        weight = weight + k.unsqueeze(1) * w3[..., None, None]\n        bias = bias + self.edge_bias + b3 * k.sum((1, 2))\n        return conv3x3(weight, bias)\n',
    'experiments/c1_rep_branches/rep_branches/model.py': '"""NuSE-E với trunk dùng conv ba nhánh. Sau `fuse()` model giống hệt e80: cùng tham số, FLOPs và latency."""\nfrom functools import partial\n\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom nuse_e.models.nuse_e import NuSEE\n\nfrom .branches import ACBConv, ECBConv\n\nCONVS = {"acb": ACBConv, "ecb": ECBConv}\n\n\nclass BranchResBlock(nn.Module):\n    """ResBlock của NuSE-E (Conv3x3 -> ReLU -> Conv3x3, skip identity), mỗi Conv3x3 là một conv ba nhánh."""\n\n    def __init__(self, width, conv):\n        super().__init__()\n        self.conv1 = conv(width)\n        self.conv2 = conv(width)\n\n    def forward(self, x):\n        return x + self.conv2(F.relu(self.conv1(x)))\n\n    def fuse(self):\n        for name in ("conv1", "conv2"):\n            conv = getattr(self, name)\n            if not isinstance(conv, nn.Conv2d):\n                setattr(self, name, conv.fuse())\n\n\nclass RepBranchNuSEE(NuSEE):\n    """arch["block"] chọn kiểu conv ba nhánh ("acb", "ecb"); arch["options"]["expand"] cho ECB (mặc định 2).\n\n    Stage, head, exit và loss giữ nguyên NuSEE; chỉ thay block của trunk. `params_for` của NuSEE đã tính block ở dạng\n    gộp (bằng Conv3x3 thường), nên các ràng buộc tham số và latency là của model sau khi gộp.\n    """\n\n    def __init__(self, stats, arch, channels=3):\n        super().__init__(stats, arch, channels)\n        conv = CONVS[arch["block"]]\n        if arch["block"] == "ecb":\n            conv = partial(conv, expand=arch.get("options", {}).get("expand", 2))\n        self.stages = nn.ModuleList(nn.Sequential(*[BranchResBlock(self.width, conv) for _ in range(n)])\n                                    for n in arch["blocks"])\n\n    def train_params(self):\n        """Số tham số lúc train (trước khi gộp), để báo cáo chi phí train."""\n        return sum(p.numel() for p in self.parameters())\n\n',
    'experiments/c1_rep_branches/test_rep_branches.py': '"""Kiểm cách 1 không cần dữ liệu: gộp nhánh cho output giống hệt, và model đã gộp chính là e80.\n\nChạy: python -m unittest experiments/c1_rep_branches/test_rep_branches.py   (từ gốc repo)\n"""\nimport sys\nimport unittest\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\nsys.path[:0] = [str(HERE.parents[1]), str(HERE)]\n\nimport torch  # noqa: E402\n\nfrom nuse_e.models.nuse_e import NuSEE  # noqa: E402\nfrom nuse_e.models.registry import make_nuse  # noqa: E402\nfrom nuse_e.nullspace.basis import helmert  # noqa: E402\nfrom nuse_e.operators import degrade, make_operator  # noqa: E402\nfrom rep_branches import ACBConv, ECBConv  # noqa: E402\n\nIMPL = "rep_branches.model:RepBranchNuSEE"\n\n\ndef fake_stats(channels=3, d=15, seed=0):\n    """Thống kê PCA giả, đủ khoá để dựng NuSEE (average pool)."""\n    g = torch.Generator().manual_seed(seed)\n    return {"method": "average_pool", "Q": helmert(),\n            "components": torch.linalg.qr(torch.randn(channels, d, d, generator=g, dtype=torch.float64))[0],\n            "mean": torch.randn(channels, d, generator=g, dtype=torch.float64) * 1e-2,\n            "eigenvalues": torch.linspace(1e-3, 5e-5, d, dtype=torch.float64).repeat(channels, 1),\n            "wiener": torch.randn(channels, 10, d, generator=g, dtype=torch.float64) * 1e-2}\n\n\ndef small_arch(block, blocks=(2, 1, 1), width=8):\n    """Cùng dạng arch_e80.json (exit 5/9/15, nhóm 5/4/6, L_g chỉ ở nhóm 1) nhưng nhỏ để chạy nhanh."""\n    arch = {"budgets": [5, 9, 15], "groups": [5, 4, 6], "blocks": list(blocks), "linear": [True, False, False],\n            "method": "average_pool", "width": width, "block": block, "alpha": "uniform"}\n    if block in ("acb", "ecb"):\n        arch["impl"], arch["options"] = IMPL, {}\n    return arch\n\n\ndef randomize(model, std=0.3, seed=0):\n    torch.manual_seed(seed)\n    with torch.no_grad():\n        for p in model.parameters():\n            p.normal_(0, std)\n    return model\n\n\nclass BranchFuseTest(unittest.TestCase):\n    def test_fused_conv_matches_three_branches_including_borders(self):\n        x = torch.randn(2, 6, 11, 13, dtype=torch.float64)      # cạnh lẻ, không vuông: kiểm cả viền\n        for conv in (ACBConv(6), ECBConv(6), ECBConv(6, expand=3)):\n            conv = randomize(conv.double())\n            with torch.no_grad():\n                err = (conv(x) - conv.fuse()(x)).abs().max().item()\n            self.assertLess(err, 1e-10, type(conv).__name__)\n\n\nclass ModelTest(unittest.TestCase):\n    def test_fused_model_is_e80_for_every_budget(self):\n        stats = fake_stats()\n        y = degrade(torch.rand(1, 3, 40, 56, dtype=torch.float64), "average_pool")\n        for block in ("acb", "ecb"):\n            model = randomize(make_nuse(stats, small_arch(block)).double().eval())\n            with torch.no_grad():\n                before = [model(y, k) for k in model.budgets]\n                model.fuse()\n                after = [model(y, k) for k in model.budgets]\n                plain = NuSEE(stats, small_arch("plain")).double().eval()\n                plain.load_state_dict(model.state_dict())        # strict: cùng tên, cùng kích thước với e80\n                as_e80 = [plain(y, k) for k in plain.budgets]\n            for k, a, b, c in zip(model.budgets, before, after, as_e80):\n                self.assertLess((a - b).abs().max().item(), 1e-9, f"{block} K={k}")\n                self.assertTrue(torch.equal(b, c), f"{block} K={k}")\n            self.assertEqual(sum(p.numel() for p in model.parameters()), model.params_for(model.budgets[-1]))\n            self.assertEqual([model.params_for(k) for k in model.budgets],\n                             [plain.params_for(k) for k in plain.budgets])\n\n    def test_data_consistency_and_nesting(self):\n        stats, op = fake_stats(), make_operator("average_pool")\n        y = degrade(torch.rand(1, 3, 32, 48, dtype=torch.float64), "average_pool")\n        model = randomize(make_nuse(stats, small_arch("ecb")).double().eval())\n        with torch.no_grad():\n            for k in model.budgets:                               # như T5: basis lưu fp32 nên so tương đối với max|z|\n                z = model.detail(model.coefficients(y, k), k)\n                err = (op.forward(model(y, k)) - y).abs().max() / z.abs().max().clamp_min(1)\n                self.assertLess(err.item(), 1e-6)\n            before = model(y, 5).clone()\n            for module in [*model.stages[1:], *model.heads[1:]]:\n                for p in module.parameters():\n                    p.add_(1.0)\n            self.assertTrue(torch.equal(before, model(y, 5)))     # exit K=5 không đọc stage / head sau\n\n    def test_training_step_runs(self):\n        model = make_nuse(fake_stats(), small_arch("ecb"))\n        hr = torch.rand(2, 3, 32, 32)\n        loss = model.loss(degrade(hr, "average_pool"), hr)\n        loss.backward()\n        self.assertTrue(torch.isfinite(loss))\n        self.assertTrue(all(p.grad is not None for p in model.stages.parameters()))\n\n\nif __name__ == "__main__":\n    unittest.main()\n',
}

for name, text in FILES.items():
    path = CODE / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
print(f"Đã ghi {len(FILES)} file vào {CODE}")


In [ ]:
import subprocess
import sys

try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips", "torchmetrics", "torch-fidelity"],
                   check=False, timeout=600)
except Exception as error:                     # không có Internet: bước evaluate tự bỏ LPIPS / FID
    print("Không cài được gói LPIPS / FID:", error)


def stream(cmd):
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(list(map(str, cmd)), cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"Lệnh lỗi (mã {proc.returncode}): {' '.join(map(str, cmd))}")


def run(*steps, extra=()):
    """python -m nuse_e.pipeline <bước...> --out OUT; dừng notebook nếu lỗi."""
    stream([sys.executable, "-u", "-m", "nuse_e.pipeline", *steps, "--out", OUT, *extra, *(["--smoke"] if SMOKE else [])])


def unit_tests():
    for test in sorted((CODE / "experiments" / PROJECT).glob("test_*.py")):
        stream([sys.executable, "-m", "unittest", test.relative_to(CODE)])


## 1. Dữ liệu, cache, kết quả phiên trước

In [ ]:
import json
import shutil


def progress(run_dir):
    """(xong chưa, số epoch đã train) của một thư mục run."""
    hist = run_dir / "history.json"
    return (run_dir / "DONE").exists(), len(json.loads(hist.read_text())) if hist.exists() else 0


# Output cũ trong Input: output v3 và output các version trước của chính notebook này. Lấy bản tiến xa nhất của từng run.
PREVIOUS = sorted({Path(d).parents[2] for d, _, files in os.walk(INPUT, followlinks=True)
                   if "last.pt" in files and Path(d).parents[1].name == "runs"})
for root in PREVIOUS:
    if (root / "stats").is_dir():
        shutil.copytree(root / "stats", OUT / "stats", dirs_exist_ok=True)
    for src in sorted((root / "runs").glob("*/*")):
        dst = OUT / src.relative_to(root)
        if (src / "last.pt").exists() and progress(src) > progress(dst):
            shutil.rmtree(dst, ignore_errors=True)
            shutil.copytree(src, dst)
            print(f"dùng {src} (xong: {progress(src)[0]}, {progress(src)[1]} epoch)")
print("Output cũ dùng lại:", [str(p) for p in PREVIOUS] or "không có (train mọi run, kể cả run tham chiếu)")
run("setup", extra=["--data-root", INPUT, "--old-runs", INPUT, "--cache", CACHE,
                    "--experiment", CODE / "experiments" / PROJECT])


## 2. Test của mini project, thống kê PCA, test chấp nhận, sàng tham số / latency

In [ ]:
unit_tests()
run("stats", "test", "screen")

## 3. Train

Mỗi GPU một run. Run có file `DONE` không train lại; run dở train tiếp từ `last.pt`. Gần hết giờ thì dừng sau epoch đang chạy (xem `TRAIN_HOURS`). Xem tab Logs nếu ô này chạy lâu.

In [ ]:
run("train", extra=[*(["--gpus", GPUS] if GPUS else []), "--deadline", START + TRAIN_HOURS * 3600])
PENDING = json.loads((OUT / "comparison" / "train_pending.json").read_text())
TRAINED = not PENDING
if PENDING:
    print("\nTRAIN CHƯA XONG, còn:", PENDING)
    print("Bỏ qua bước đánh giá. Add Input = output của version này, rồi Save & Run All lần nữa để train tiếp.")


## 4. Chọn, đánh giá, chi phí, báo cáo, hình (bỏ qua nếu train chưa xong)

In [ ]:
if TRAINED:
    run("curves", "select", "evaluate", "cost", "report", "figures")

## 5. Kết quả của mini project

In [ ]:
if TRAINED:
    import csv

    rows = list(csv.DictReader(open(OUT / "comparison" / "variants.csv")))
    refs = {r["lr"]: r for r in rows if r["model"] == "NuSE-E/e80"}                # e80 cùng loại LR (K khác nhau giữa hai loại)
    ks = sorted({int(k[6:]) for r in rows for k in r if k.startswith("psnr_K") and r[k]})
    print("PSNR DIV2K valid (100 ảnh), chênh so với e80 cùng loại LR trong ngoặc. Hai lần train cùng seed lệch nhau tới 0.01 dB.")
    print(f"{'LR':12s} {'model':16s} {'tham số K max':>13s} " + " ".join(f"{'K=' + str(k):>17s}" for k in ks))
    for r in rows:
        ref = refs.get(r["lr"])
        cols = []
        for k in ks:
            v = r.get(f"psnr_K{k}")
            if not v:
                cols.append(f"{'—':>17s}")
                continue
            d = f"({float(v) - float(ref[f'psnr_K{k}']):+.3f})" if ref and ref.get(f"psnr_K{k}") else ""
            cols.append(f"{float(v):8.3f} {d:>8s}")
        print(f"{r['lr']:12s} {r['model']:16s} {int(r['params_kmax']):>13,d} " + " ".join(cols))

    try:
        from IPython.display import Image, Markdown, display
        display(Markdown((OUT / "comparison" / "report.md").read_text()))
        for png in sorted((OUT / "comparison" / "figures").glob("*.png")):
            display(Image(str(png)))
    except ImportError:                              # chạy như script ngoài Jupyter
        print((OUT / "comparison" / "report.md").read_text())
